In [1]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  MaD-CoRN: Deepfake Detection    
# ║
# ║  Paper: "MaD-CoRN: an efficient and lightweight deepfake detection          ║
# ║          approach using convolutional reservoir network"                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
#
# USAGE IN JUPYTER
# ────────────────
# 1. pip install torch torchvision scikit-learn pillow numpy pandas
# 2. Set CONFIG below (data_dir, epochs, batch_size, etc.)
# 3. Run All Cells
#
# DATASET LAYOUT expected at data_dir:
#   celeb_df/
#       real/          ← or Celeb-real/, YouTube-real/
#           frame_001.jpg  ...
#       fake/          ← or Celeb-synthesis/
#           frame_001.jpg  ...
#
# OR pass csv_path pointing to a CSV with columns: path, label (0=real 1=fake)

In [1]:
#FF++ with balance configuration -- oversampling real data with augmentation techniques
!pip install scikit-learn torch torchvision tqdm pandas pillow

import os, io, json, math, time, random, warnings
import numpy as np
import pandas as pd
from dataclasses import dataclass, asdict
from PIL import Image, ImageFile

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torchvision.transforms as T
from torch.cuda.amp import GradScaler, autocast

from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, accuracy_score

# ============================================================
# CONFIG
# ============================================================

CONFIG = dict(
    metadata_csv="/home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/metadata.csv",

    resolutions=[128, 224, 256, 384],
    image_size_for_model=224,

    epochs=15,
    batch_size=32,
    lr=1e-3,
    weight_decay=1e-4,
    label_smoothing=0.05,
    warmup_epochs=3,

    base_ch=32,

    num_workers=0,
    seed=42,
    save_dir="./mad_corn_results_ffpp_balanced",
    device="auto",
    smoke_test=False,
)

# ============================================================
# HELPERS
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def auto_device(requested):
    if requested != "auto":
        return torch.device(requested)
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

set_seed(CONFIG["seed"])
DEVICE = auto_device(CONFIG["device"])
os.makedirs(CONFIG["save_dir"], exist_ok=True)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", DEVICE)

# ============================================================
# RESULT CLASSES
# ============================================================

@dataclass
class EpochResult:
    epoch: int
    train_loss: float
    train_acc: float
    train_auc: float
    val_acc: float
    val_auc: float
    precision: float
    recall: float
    f1: float
    uncertain_rate: float
    lr: float

@dataclass
class TestResult:
    model_name: str
    accuracy: float
    auc: float
    precision: float
    recall: float
    f1: float
    uncertain_rate: float

@dataclass
class ResolutionResult:
    resolution: int
    accuracy: float
    auc: float
    fast_pct: float
    medium_pct: float
    full_pct: float

@dataclass
class InferenceResult:
    mode: str
    time_per_image_ms: float
    fps: float

@dataclass
class TrainingTimeResult:
    model_type: str
    minutes: float
    hours: float

# ============================================================
# MODEL
# ============================================================

class ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3, stride=1, padding=1, groups=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel, stride, padding, groups=groups, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU6(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class DepthwiseSeparable(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.dw = ConvBNReLU(in_ch, in_ch, kernel=3, stride=stride, padding=1, groups=in_ch)
        self.pw = ConvBNReLU(in_ch, out_ch, kernel=1, padding=0)

    def forward(self, x):
        return self.pw(self.dw(x))

class ConvReservoir(nn.Module):
    def __init__(self, in_ch, reservoir_ch=64, spectral_radius=0.9):
        super().__init__()
        self.input_proj = nn.Conv2d(in_ch, reservoir_ch, kernel_size=1, bias=False)
        self.input_bn = nn.BatchNorm2d(reservoir_ch)

        W = torch.randn(reservoir_ch, reservoir_ch, 3, 3)
        W = W / (W.abs().max() + 1e-8) * spectral_radius
        self.register_buffer("W_res", W)

        self.readout = nn.Sequential(
            nn.Conv2d(reservoir_ch, reservoir_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(reservoir_ch),
            nn.ReLU6(inplace=True),
        )

    def forward(self, x):
        h = F.relu(self.input_bn(self.input_proj(x)))
        h_res = torch.tanh(F.conv2d(h, self.W_res, padding=1))
        return self.readout(h + h_res)

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, mid),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        w = self.gate(x).view(x.size(0), x.size(1), 1, 1)
        return x * w

class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx, _ = x.max(dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))

class MultiScaleAttentionBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        b_ch = out_ch // 3

        self.branch1 = DepthwiseSeparable(in_ch, b_ch)
        self.branch2 = nn.Sequential(
            DepthwiseSeparable(in_ch, b_ch),
            DepthwiseSeparable(b_ch, b_ch),
        )
        self.branch3 = nn.Sequential(
            DepthwiseSeparable(in_ch, b_ch),
            DepthwiseSeparable(b_ch, b_ch),
            DepthwiseSeparable(b_ch, b_ch),
        )

        self.fuse = ConvBNReLU(b_ch * 3, out_ch, kernel=1, padding=0)
        self.ca = ChannelAttention(out_ch)
        self.sa = SpatialAttention()

    def forward(self, x):
        f = self.fuse(torch.cat([self.branch1(x), self.branch2(x), self.branch3(x)], dim=1))
        return self.sa(self.ca(f))

class MaDCoRN(nn.Module):
    def __init__(self, in_ch=3, num_classes=2, base_ch=32):
        super().__init__()
        c = base_ch

        self.stem = nn.Sequential(
            ConvBNReLU(in_ch, c, kernel=3, stride=2, padding=1),
            ConvBNReLU(c, c * 2, kernel=3, stride=1, padding=1),
        )

        self.mad1 = MultiScaleAttentionBlock(c * 2, c * 4)
        self.mad2 = MultiScaleAttentionBlock(c * 4, c * 4)
        self.res1 = ConvReservoir(c * 4, c * 4)
        self.down1 = nn.Conv2d(c * 4, c * 4, kernel_size=3, stride=2, padding=1, bias=False)

        self.mad3 = MultiScaleAttentionBlock(c * 4, c * 8)
        self.mad4 = MultiScaleAttentionBlock(c * 8, c * 8)
        self.res2 = ConvReservoir(c * 8, c * 8)
        self.down2 = nn.Conv2d(c * 8, c * 8, kernel_size=3, stride=2, padding=1, bias=False)

        self.mad5 = MultiScaleAttentionBlock(c * 8, c * 16)
        self.mad6 = MultiScaleAttentionBlock(c * 16, c * 16)
        self.res3 = ConvReservoir(c * 16, c * 16)

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(c * 16, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x)
        x = self.down1(self.res1(self.mad2(self.mad1(x))))
        x = self.down2(self.res2(self.mad4(self.mad3(x))))
        x = self.res3(self.mad6(self.mad5(x)))
        return self.head(self.pool(x))

    def count_parameters(self):
        total = sum(p.numel() for p in self.parameters())
        frozen = sum(p.numel() for p in self.parameters() if not p.requires_grad)
        return total, frozen

def freeze_reservoir_weights(model):
    for name, param in model.named_parameters():
        if "W_res" in name:
            param.requires_grad_(False)

# ============================================================
# DATASET
# ============================================================

class AddGaussianNoise:
    def __init__(self, std=0.01):
        self.std = std

    def __call__(self, t):
        return t + torch.randn_like(t) * self.std

class JPEGCompression:
    def __init__(self, quality_range=(60, 95)):
        self.lo, self.hi = quality_range

    def __call__(self, img):
        q = random.randint(self.lo, self.hi)
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=q)
        buf.seek(0)
        return Image.open(buf).copy()

_MEAN = [0.485, 0.456, 0.406]
_STD = [0.229, 0.224, 0.225]

def build_transforms(source_resolution, split="train", image_size_for_model=224):
    if split == "train":
        return T.Compose([
            T.Resize((source_resolution, source_resolution)),
            T.Resize((image_size_for_model + 16, image_size_for_model + 16)),
            T.RandomCrop(image_size_for_model),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.1),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
            T.RandomGrayscale(p=0.05),
            T.RandomApply([T.GaussianBlur(kernel_size=3)], p=0.15),
            JPEGCompression(quality_range=(60, 95)),
            T.ToTensor(),
            AddGaussianNoise(std=0.01),
            T.Normalize(_MEAN, _STD),
        ])

    return T.Compose([
        T.Resize((source_resolution, source_resolution)),
        T.Resize((image_size_for_model, image_size_for_model)),
        T.ToTensor(),
        T.Normalize(_MEAN, _STD),
    ])

def load_ffpp_dataframes(cfg):
    df = pd.read_csv(cfg["metadata_csv"])

    required = {"frame_path", "split", "label", "label_name", "video_id"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"metadata.csv missing columns: {missing}")

    df = df[df["frame_path"].apply(os.path.exists)].reset_index(drop=True)

    print("\nLoaded FF++ metadata")
    print("Total usable frames:", len(df))

    for split in ["Train", "Val", "Test"]:
        sdf = df[df["split"].str.lower() == split.lower()]
        real = int((sdf["label"] == 0).sum())
        fake = int((sdf["label"] == 1).sum())
        videos = sdf["video_id"].nunique()
        print(f"{split}: real={real}, fake={fake}, total={len(sdf)}, videos={videos}")

    train_full = df[df["split"].str.lower() == "train"].reset_index(drop=True)
    val_df = df[df["split"].str.lower() == "val"].reset_index(drop=True)
    test_df = df[df["split"].str.lower() == "test"].reset_index(drop=True)

    return train_full, val_df, test_df

def make_oversampled_train_df(train_df, seed=42):
    real_df = train_df[train_df["label"] == 0].copy()
    fake_df = train_df[train_df["label"] == 1].copy()

    real_os = real_df.sample(
        n=len(fake_df),
        replace=True,
        random_state=seed
    ).copy()

    real_os["is_oversampled_real"] = True
    fake_df["is_oversampled_real"] = False

    oversampled_df = (
        pd.concat([real_os, fake_df])
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )

    print("\nOversampled train set")
    print(oversampled_df["label_name"].value_counts())
    print("Total oversampled train samples:", len(oversampled_df))
    print("Oversampled real rows:", int(oversampled_df["is_oversampled_real"].sum()))

    return oversampled_df

def make_balanced_train_df(train_df, seed=42):
    real_df = train_df[train_df["label"] == 0]
    fake_df = train_df[train_df["label"] == 1]

    n = min(len(real_df), len(fake_df))

    real_bal = real_df.sample(n=n, random_state=seed)
    fake_bal = fake_df.sample(n=n, random_state=seed)

    balanced_df = (
        pd.concat([real_bal, fake_bal])
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )

    print("\nBalanced train set by undersampling")
    print(balanced_df["label_name"].value_counts())
    print("Total balanced train samples:", len(balanced_df))

    return balanced_df

def df_to_samples(df):
    return [(row["frame_path"], int(row["label"])) for _, row in df.iterrows()]

class MultiResolutionFFPPDataset(Dataset):
    def __init__(self, base_samples, resolutions, split="train", image_size_for_model=224):
        self.base_samples = list(base_samples)
        self.resolutions = list(resolutions)
        self.split = split
        self.image_size_for_model = image_size_for_model

        self.transforms = {
            res: build_transforms(res, split=split, image_size_for_model=image_size_for_model)
            for res in self.resolutions
        }

        if split == "train":
            self.samples = [(path, label, None) for path, label in self.base_samples]
        else:
            self.samples = [
                (path, label, res)
                for path, label in self.base_samples
                for res in self.resolutions
            ]

        n_real = sum(1 for _, label in self.base_samples if label == 0)
        n_fake = len(self.base_samples) - n_real

        print(
            f"[{split}] base: real={n_real:,}, fake={n_fake:,}, total={len(self.base_samples):,} | "
            f"effective={len(self.samples):,}"
        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label, resolution = self.samples[idx]

        if resolution is None:
            resolution = random.choice(self.resolutions)

        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            img = Image.fromarray(np.zeros((resolution, resolution, 3), dtype=np.uint8))

        x = self.transforms[resolution](img)
        y = torch.tensor(label, dtype=torch.long)

        return x, y

    def subset_for_resolution(self, resolution):
        return SingleResolutionView(
            self.base_samples,
            resolution,
            image_size_for_model=self.image_size_for_model,
        )

    @property
    def base_count(self):
        return len(self.base_samples)

    @property
    def expanded_count(self):
        return len(self.samples)

class SingleResolutionView(Dataset):
    def __init__(self, base_samples, resolution, image_size_for_model=224):
        self.base_samples = list(base_samples)
        self.resolution = resolution
        self.transform = build_transforms(
            resolution,
            split="val",
            image_size_for_model=image_size_for_model,
        )

    def __len__(self):
        return len(self.base_samples)

    def __getitem__(self, idx):
        path, label = self.base_samples[idx]

        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            img = Image.fromarray(np.zeros((self.resolution, self.resolution, 3), dtype=np.uint8))

        x = self.transform(img)
        y = torch.tensor(label, dtype=torch.long)

        return x, y

def _make_loader(dataset, batch_size, num_workers, shuffle, drop_last):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=False,
    )

def print_dataset_summary(split_datasets, resolutions):
    print("\nDataset size summary")
    for name in ["train", "val", "test"]:
        ds = split_datasets[name]
        print(
            f"{name.title():<5} | base={ds.base_count:,}, "
            f"effective={ds.expanded_count:,}, resolutions={resolutions}"
        )

    print("\nStrategy:")
    print("  Train: balanced undersampled train set + one random source resolution per image per epoch")
    print("  Val:   original val set + all requested source resolutions per image")
    print("  Test:  original test set + all requested source resolutions per image")
    print(f"  Model input tensor size: (3, {CONFIG['image_size_for_model']}, {CONFIG['image_size_for_model']})")

def make_dataloaders(cfg):
    train_full_df, val_df, test_df = load_ffpp_dataframes(cfg)
    train_balanced_df = make_oversampled_train_df(train_full_df, seed=cfg["seed"])
    train_samples = df_to_samples(train_balanced_df)
    val_samples = df_to_samples(val_df)
    test_samples = df_to_samples(test_df)

    resolutions = list(cfg["resolutions"])

    split_datasets = {
        "train": MultiResolutionFFPPDataset(
            train_samples,
            resolutions,
            split="train",
            image_size_for_model=cfg["image_size_for_model"],
        ),
        "val": MultiResolutionFFPPDataset(
            val_samples,
            resolutions,
            split="val",
            image_size_for_model=cfg["image_size_for_model"],
        ),
        "test": MultiResolutionFFPPDataset(
            test_samples,
            resolutions,
            split="test",
            image_size_for_model=cfg["image_size_for_model"],
        ),
    }

    print_dataset_summary(split_datasets, resolutions)

    loaders = {
        "train": _make_loader(
            split_datasets["train"],
            cfg["batch_size"],
            cfg["num_workers"],
            shuffle=True,
            drop_last=True,
        ),
        "val": _make_loader(
            split_datasets["val"],
            cfg["batch_size"],
            cfg["num_workers"],
            shuffle=False,
            drop_last=False,
        ),
        "test": _make_loader(
            split_datasets["test"],
            cfg["batch_size"],
            cfg["num_workers"],
            shuffle=False,
            drop_last=False,
        ),
    }

    meta = {
        "train_full_df": train_full_df,
        "train_balanced_df": train_balanced_df,
        "val_df": val_df,
        "test_df": test_df,
    }

    return {
        "loaders": loaders,
        "datasets": split_datasets,
        "meta": meta,
    }

def make_smoke_loader(resolution, n=200, bs=8):
    imgs = torch.randn(n, 3, CONFIG["image_size_for_model"], CONFIG["image_size_for_model"])
    labels = torch.randint(0, 2, (n,))
    return DataLoader(TensorDataset(imgs, labels), batch_size=bs, shuffle=True)

# ============================================================
# TRAINING
# ============================================================

def compute_uncertain_rate(probs, threshold=0.65):
    conf = probs.max(axis=1)
    return float(np.mean(conf < threshold) * 100)

def run_epoch(model, loader, criterion, optimizer, device, scaler=None, is_train=True, desc=None):
    model.train(is_train)

    total_loss = 0.0
    seen = 0
    all_probs, all_preds, all_labels = [], [], []

    pbar = tqdm(
        loader,
        total=len(loader) if hasattr(loader, "__len__") else None,
        desc=desc or ("Train" if is_train else "Eval"),
        leave=False,
        dynamic_ncols=True,
        mininterval=0.5,
    )

    with torch.set_grad_enabled(is_train):
        for imgs, labels in pbar:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            if is_train:
                optimizer.zero_grad(set_to_none=True)

            if is_train and scaler:
                with autocast():
                    logits = model(imgs)
                    loss = criterion(logits, labels)

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()

            else:
                logits = model(imgs)
                loss = criterion(logits, labels)

                if is_train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()

            probs = torch.softmax(logits.float(), dim=1).detach().cpu().numpy()
            preds = probs.argmax(axis=1)

            all_probs.append(probs)
            all_preds.append(preds)
            all_labels.append(labels.detach().cpu().numpy())

            batch_n = len(labels)
            total_loss += loss.item() * batch_n
            seen += batch_n

            pbar.set_postfix(loss=f"{loss.item():.4f}", samples=f"{seen:,}")

    all_probs = np.vstack(all_probs)
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    acc = accuracy_score(all_labels, all_preds)

    try:
        auc = roc_auc_score(all_labels, all_probs[:, 1])
    except ValueError:
        auc = 0.5

    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    uncertain_rate = compute_uncertain_rate(all_probs)

    return {
        "loss": total_loss / max(seen, 1),
        "acc": acc,
        "auc": auc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "uncertain_rate": uncertain_rate,
        "probs": all_probs,
        "preds": all_preds,
        "labels": all_labels,
    }

class Trainer:
    def __init__(self, model, train_loader, val_loader, test_loader, device, cfg, ckpt_path):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader
        self.device = device
        self.epochs = cfg["epochs"]
        self.ckpt_path = ckpt_path
        self.model_name = "MaD-CoRN-FFPP-Balanced"

        self.criterion = nn.CrossEntropyLoss(label_smoothing=cfg["label_smoothing"])

        self.optimizer = optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=cfg["lr"],
            weight_decay=cfg["weight_decay"],
        )

        total_steps = self.epochs
        warmup_steps = cfg["warmup_epochs"]

        def lr_lambda(epoch):
            if epoch < warmup_steps:
                return max(1e-8, epoch / max(1, warmup_steps))
            prog = (epoch - warmup_steps) / max(1, total_steps - warmup_steps)
            return 0.5 * (1.0 + math.cos(math.pi * prog))

        self.scheduler = optim.lr_scheduler.LambdaLR(self.optimizer, lr_lambda)
        self.scaler = GradScaler() if device.type == "cuda" else None
        self.epoch_results = []

    def train(self):
        best_auc = -1.0
        t0 = time.time()

        for epoch in range(1, self.epochs + 1):
            tr = run_epoch(
                self.model,
                self.train_loader,
                self.criterion,
                self.optimizer,
                self.device,
                self.scaler,
                is_train=True,
                desc=f"Epoch {epoch}/{self.epochs} [train]",
            )

            with torch.no_grad():
                vl = run_epoch(
                    self.model,
                    self.val_loader,
                    self.criterion,
                    None,
                    self.device,
                    None,
                    is_train=False,
                    desc=f"Epoch {epoch}/{self.epochs} [val]",
                )

            self.scheduler.step()
            lr_now = self.optimizer.param_groups[0]["lr"]

            er = EpochResult(
                epoch=epoch,
                train_loss=tr["loss"],
                train_acc=tr["acc"],
                train_auc=tr["auc"],
                val_acc=vl["acc"],
                val_auc=vl["auc"],
                precision=vl["precision"],
                recall=vl["recall"],
                f1=vl["f1"],
                uncertain_rate=vl["uncertain_rate"],
                lr=lr_now,
            )

            self.epoch_results.append(er)

            print(
                f"Epoch {epoch:3d}/{self.epochs} | "
                f"Loss {er.train_loss:.4f} | "
                f"Tr Acc {er.train_acc:.4f} AUC {er.train_auc:.4f} | "
                f"Val Acc {er.val_acc:.4f} AUC {er.val_auc:.4f} | "
                f"P {er.precision:.4f} R {er.recall:.4f} F1 {er.f1:.4f} | "
                f"Unc {er.uncertain_rate:.1f}% | LR {lr_now:.2e}"
            )

            if vl["auc"] > best_auc:
                best_auc = vl["auc"]
                torch.save(self.model.state_dict(), self.ckpt_path)
                print(f"Checkpoint saved. Val AUC = {best_auc:.4f}")

        return self.epoch_results, (time.time() - t0) / 60

    def evaluate(self, loader=None, load_best=True):
        if load_best and os.path.exists(self.ckpt_path):
            self.model.load_state_dict(torch.load(self.ckpt_path, map_location=self.device))

        loader = loader or self.test_loader

        with torch.no_grad():
            r = run_epoch(
                self.model,
                loader,
                self.criterion,
                None,
                self.device,
                None,
                is_train=False,
                desc="Test evaluation",
            )

        return TestResult(
            model_name=self.model_name,
            accuracy=r["acc"],
            auc=r["auc"],
            precision=r["precision"],
            recall=r["recall"],
            f1=r["f1"],
            uncertain_rate=r["uncertain_rate"],
        )

# ============================================================
# EVAL / TIMING
# ============================================================

@torch.no_grad()
def measure_inference(model, resolution, device, image_size_for_model=224, batch_size=32, n_warmup=20, n_runs=200):
    model.eval()
    results = []

    for bs, label in [(1, "Single"), (batch_size, f"Batch={batch_size}")]:
        dummy = torch.randn(bs, 3, image_size_for_model, image_size_for_model, device=device)

        for _ in range(n_warmup):
            model(dummy)

        if device.type == "cuda":
            torch.cuda.synchronize()

        t0 = time.perf_counter()

        for _ in range(n_runs):
            model(dummy)

        if device.type == "cuda":
            torch.cuda.synchronize()

        elapsed = time.perf_counter() - t0
        total = bs * n_runs

        results.append(
            InferenceResult(
                mode=f"Source Res={resolution} {label}",
                time_per_image_ms=elapsed / total * 1000,
                fps=total / elapsed,
            )
        )

    return results

@torch.no_grad()
def evaluate_by_resolution(model, loader, device, resolution):
    model.eval()

    all_probs, all_labels = [], []
    buckets = {"fast": 0, "medium": 0, "full": 0}
    total = 0

    pbar = tqdm(
        loader,
        total=len(loader) if hasattr(loader, "__len__") else None,
        desc=f"Resolution {resolution} eval",
        leave=False,
        dynamic_ncols=True,
        mininterval=0.5,
    )

    for imgs, labels in pbar:
        imgs = imgs.to(device, non_blocking=True)
        bs = imgs.size(0)

        if device.type == "cuda":
            torch.cuda.synchronize()

        t0 = time.perf_counter()
        logits = model(imgs)

        if device.type == "cuda":
            torch.cuda.synchronize()

        ms = (time.perf_counter() - t0) / bs * 1000

        probs = torch.softmax(logits.float(), dim=1).cpu().numpy()

        all_probs.append(probs)
        all_labels.append(labels.numpy())

        if ms < 5:
            buckets["fast"] += bs
        elif ms < 20:
            buckets["medium"] += bs
        else:
            buckets["full"] += bs

        total += bs
        pbar.set_postfix(samples=f"{total:,}", ms_per_img=f"{ms:.3f}")

    all_probs = np.vstack(all_probs)
    all_labels = np.concatenate(all_labels)
    preds = all_probs.argmax(axis=1)

    acc = accuracy_score(all_labels, preds)

    try:
        auc = roc_auc_score(all_labels, all_probs[:, 1])
    except ValueError:
        auc = 0.5

    return ResolutionResult(
        resolution=resolution,
        accuracy=acc,
        auc=auc,
        fast_pct=buckets["fast"] / max(total, 1) * 100,
        medium_pct=buckets["medium"] / max(total, 1) * 100,
        full_pct=buckets["full"] / max(total, 1) * 100,
    )

# ============================================================
# REPORTING
# ============================================================

def _table(headers, rows, title=""):
    col_w = [
        max(len(str(h)), max((len(str(r[i])) for r in rows), default=0))
        for i, h in enumerate(headers)
    ]

    sep = "+" + "+".join("-" * (w + 2) for w in col_w) + "+"
    hdr = "|" + "|".join(f" {h:<{w}} " for h, w in zip(headers, col_w)) + "|"

    body = [
        "|" + "|".join(f" {str(v):<{w}} " for v, w in zip(r, col_w)) + "|"
        for r in rows
    ]

    out = [sep, hdr, sep] + body + [sep]

    if title:
        out.insert(0, title.center(len(sep)))
        out.insert(1, "")

    return "\n".join(out)

def print_training_table(ers):
    h = [
        "Epoch", "Train Loss", "Train Acc", "Train AUC",
        "Val Acc", "Val AUC", "Precision", "Recall", "F1",
        "Uncertain Rate (%)"
    ]

    rows = [
        [
            e.epoch,
            f"{e.train_loss:.4f}",
            f"{e.train_acc:.4f}",
            f"{e.train_auc:.4f}",
            f"{e.val_acc:.4f}",
            f"{e.val_auc:.4f}",
            f"{e.precision:.4f}",
            f"{e.recall:.4f}",
            f"{e.f1:.4f}",
            f"{e.uncertain_rate:.2f}",
        ]
        for e in ers
    ]

    t = _table(h, rows, "Training Periods Results")
    print(t)
    return t

def print_test_table(trs):
    h = ["Model", "Accuracy", "AUC", "Precision", "Recall", "F1", "Uncertain Rate (%)"]

    rows = [
        [
            r.model_name,
            f"{r.accuracy:.4f}",
            f"{r.auc:.4f}",
            f"{r.precision:.4f}",
            f"{r.recall:.4f}",
            f"{r.f1:.4f}",
            f"{r.uncertain_rate:.2f}",
        ]
        for r in trs
    ]

    t = _table(h, rows, "Test Results")
    print(t)
    return t

def print_resolution_table(rrs):
    h = ["Resolution", "Accuracy", "AUC", "Fast (%)", "Medium (%)", "Full (%)"]

    rows = [
        [
            r.resolution,
            f"{r.accuracy:.4f}",
            f"{r.auc:.4f}",
            f"{r.fast_pct:.2f}",
            f"{r.medium_pct:.2f}",
            f"{r.full_pct:.2f}",
        ]
        for r in rrs
    ]

    t = _table(h, rows, "Accuracy Based on Resolution")
    print(t)
    return t

def print_inference_table(irs):
    h = ["Mode", "Time per Image (ms)", "FPS"]

    rows = [
        [r.mode, f"{r.time_per_image_ms:.3f}", f"{r.fps:.1f}"]
        for r in irs
    ]

    t = _table(h, rows, "Inference Time")
    print(t)
    return t

def print_training_time_table(tts):
    h = ["Model Type", "min", "hrs"]

    rows = [
        [r.model_type, f"{r.minutes:.1f}", f"{r.hours:.2f}"]
        for r in tts
    ]

    t = _table(h, rows, "Training Time")
    print(t)
    return t

def save_all_results(epoch_results, test_results, res_results, inf_results, tt_results, save_dir, meta):
    os.makedirs(save_dir, exist_ok=True)

    data = {
        "epoch_results": [asdict(e) for e in epoch_results],
        "test_results": [asdict(t) for t in test_results],
        "res_results": [asdict(r) for r in res_results],
        "inference": [asdict(i) for i in inf_results],
        "training_time": [asdict(t) for t in tt_results],
    }

    with open(os.path.join(save_dir, "mad_corn_results.json"), "w") as f:
        json.dump(data, f, indent=2)

    for name, rows in data.items():
        if rows:
            pd.DataFrame(rows).to_csv(os.path.join(save_dir, f"mad_corn_{name}.csv"), index=False)

    train_full_df = meta["train_full_df"]
    train_balanced_df = meta["train_balanced_df"]
    val_df = meta["val_df"]
    test_df = meta["test_df"]

    experiment_summary = pd.DataFrame([{
        "Experiment Name": "MaD-CoRN Balanced Train",
        "Dataset": "FaceForensics++ C23",
        "Train Samples Original": len(train_full_df),
        "Train Samples Used Balanced": len(train_balanced_df),
        "Validation Samples": len(val_df),
        "Test Samples": len(test_df),
        "Train Real Used": int((train_balanced_df["label"] == 0).sum()),
        "Train Fake Used": int((train_balanced_df["label"] == 1).sum()),
        "Validation Real": int((val_df["label"] == 0).sum()),
        "Validation Fake": int((val_df["label"] == 1).sum()),
        "Test Real": int((test_df["label"] == 0).sum()),
        "Test Fake": int((test_df["label"] == 1).sum()),
        "Balancing Method": "Train-only undersampling of fake class to match real class",
        "Resolutions Used": str(CONFIG["resolutions"]),
        "Input Modalities": "RGB",
        "RGB Shape": f"(3, {CONFIG['image_size_for_model']}, {CONFIG['image_size_for_model']})",
        "Device": str(DEVICE),
        "CUDA Available": torch.cuda.is_available(),
        "Framework": "PyTorch",
        "Torch Version": torch.__version__,
        "Balancing Method": "Train-only oversampling of real class with replacement to match fake class"
    }])

    experiment_summary.to_csv(os.path.join(save_dir, "experiment_summary.csv"), index=False)

    print("\nExperiment Summary")
    display(experiment_summary)

    print("\nResults saved to:", save_dir)

def print_full_report(epoch_results, test_results, res_results, inf_results, tt_results, save_dir, meta):
    div = "=" * 100

    print("\n" + div)
    print("MaD-CoRN EXPERIMENT RESULTS".center(100, "="))
    print(div)

    print("\n[1] TRAINING PERIODS RESULTS")
    print_training_table(epoch_results)

    print("\n[2] TEST RESULTS")
    print_test_table(test_results)

    print("\n[3] ACCURACY BASED ON RESOLUTION")
    print_resolution_table(res_results)

    print("\n[4] INFERENCE TIME")
    print_inference_table(inf_results)

    print("\n[5] TRAINING TIME")
    print_training_time_table(tt_results)

    save_all_results(epoch_results, test_results, res_results, inf_results, tt_results, save_dir, meta)

# ============================================================
# PIPELINE
# ============================================================

def run_pipeline(cfg):
    set_seed(cfg["seed"])
    device = auto_device(cfg["device"])
    os.makedirs(cfg["save_dir"], exist_ok=True)

    print("\n" + "=" * 70)
    print("MaD-CoRN Pipeline FF++ Balanced Train")
    print("Device:", device)
    print("Source resolutions:", cfg["resolutions"])
    print("Model input size:", cfg["image_size_for_model"])
    print("Epochs:", cfg["epochs"])
    print("Smoke test:", cfg["smoke_test"])
    print("=" * 70)

    model = MaDCoRN(in_ch=3, num_classes=2, base_ch=cfg["base_ch"])
    freeze_reservoir_weights(model)

    total, frozen = model.count_parameters()
    trainable = total - frozen
    print(f"\nModel params: {total:,} total | {trainable:,} trainable | {frozen:,} frozen")

    split_datasets = None
    meta = None

    if cfg["smoke_test"]:
        print("\nSMOKE TEST")
        bs = cfg["batch_size"]
        train_loader = make_smoke_loader(cfg["image_size_for_model"], n=320, bs=bs)
        val_loader = make_smoke_loader(cfg["image_size_for_model"], n=80, bs=bs)
        test_loader = make_smoke_loader(cfg["image_size_for_model"], n=80, bs=bs)
        cfg = {**cfg, "epochs": min(cfg["epochs"], 3)}
        meta = {
            "train_full_df": pd.DataFrame(),
            "train_balanced_df": pd.DataFrame(),
            "val_df": pd.DataFrame(),
            "test_df": pd.DataFrame(),
        }
    else:
        print("\nLoading FF++ dataset...")
        data_bundle = make_dataloaders(cfg)
        split_datasets = data_bundle["datasets"]
        train_loader = data_bundle["loaders"]["train"]
        val_loader = data_bundle["loaders"]["val"]
        test_loader = data_bundle["loaders"]["test"]
        meta = data_bundle["meta"]

    ckpt_path = os.path.join(cfg["save_dir"], "mad_corn_ffpp_balanced_multires.pt")

    print("\n" + "-" * 60)
    print("TRAINING")
    print("-" * 60)

    trainer = Trainer(
        model,
        train_loader,
        val_loader,
        test_loader,
        device,
        cfg,
        ckpt_path,
    )

    epoch_results, training_min = trainer.train()

    print(f"\nTraining done: {training_min:.1f} min ({training_min / 60:.2f} hrs)")

    print("\n" + "-" * 60)
    print("TEST EVALUATION ON COMBINED MULTI-RES TEST SET")
    print("-" * 60)

    test_result = trainer.evaluate(test_loader, load_best=True)
    test_results = [test_result]

    print("\n" + "-" * 60)
    print("PER-RESOLUTION BREAKDOWN")
    print("-" * 60)

    res_results = []
    inf_results = []

    for res in tqdm(cfg["resolutions"], desc="Resolution sweep", dynamic_ncols=True):
        print(f"\nResolution {res}")

        if cfg["smoke_test"]:
            res_loader = make_smoke_loader(cfg["image_size_for_model"], n=80, bs=cfg["batch_size"])
        else:
            res_ds = split_datasets["test"].subset_for_resolution(res)
            res_loader = DataLoader(
                res_ds,
                batch_size=cfg["batch_size"],
                shuffle=False,
                num_workers=cfg["num_workers"],
                pin_memory=(device.type == "cuda"),
                persistent_workers=False,
            )

        rr = evaluate_by_resolution(model, res_loader, device, res)
        res_results.append(rr)

        print(
            f"Acc={rr.accuracy:.4f} AUC={rr.auc:.4f} "
            f"Fast={rr.fast_pct:.2f}% Medium={rr.medium_pct:.2f}% Full={rr.full_pct:.2f}%"
        )

        infs = measure_inference(
            model,
            res,
            device,
            image_size_for_model=cfg["image_size_for_model"],
            batch_size=cfg["batch_size"],
        )
        inf_results.extend(infs)

        for ir in infs:
            print(f"{ir.mode}: {ir.time_per_image_ms:.3f} ms/img | {ir.fps:.1f} FPS")

    tt_results = [
        TrainingTimeResult(
            model_type="MaD-CoRN FF++ balanced train multi-resolution training",
            minutes=training_min,
            hours=training_min / 60,
        )
    ]

    print_full_report(
        epoch_results,
        test_results,
        res_results,
        inf_results,
        tt_results,
        cfg["save_dir"],
        meta,
    )

    return {
        "epoch_results": epoch_results,
        "test_results": test_results,
        "res_results": res_results,
        "inf_results": inf_results,
        "tt_results": tt_results,
        "model": model,
        "trainer": trainer,
        "split_datasets": split_datasets,
        "meta": meta,
    }

results = run_pipeline(CONFIG)

  Using cached scikit_learn-1.8.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached torch-2.11.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached torchvision-0.26.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached pandas-3.0.2-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached pillow-12.2.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.8 kB)
  Using cached numpy-2.4.4-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-

Epoch   1/15 | Loss 0.6956 | Tr Acc 0.4999 AUC 0.4990 | Val Acc 0.1429 AUC 0.5096 | P 0.0000 R 0.0000 F1 0.0000 | Unc 100.0% | LR 3.33e-04
Checkpoint saved. Val AUC = 0.5096


Epoch   2/15 | Loss 0.6846 | Tr Acc 0.5376 AUC 0.5496 | Val Acc 0.3495 AUC 0.5958 | P 0.9048 R 0.2694 F1 0.4152 | Unc 80.2% | LR 6.67e-04
Checkpoint saved. Val AUC = 0.5958


Epoch   3/15 | Loss 0.6701 | Tr Acc 0.5635 AUC 0.5746 | Val Acc 0.3206 AUC 0.5519 | P 0.9024 R 0.2326 F1 0.3698 | Unc 82.1% | LR 1.00e-03


Epoch   4/15 | Loss 0.6656 | Tr Acc 0.5666 AUC 0.5739 | Val Acc 0.2745 AUC 0.5677 | P 0.9706 R 0.1583 F1 0.2723 | Unc 89.4% | LR 9.83e-04


...

Acc=0.3539 AUC=0.5420 Fast=100.00% Medium=0.00% Full=0.00%


Resolution sweep: 100%|██████████| 4/4 [07:20<00:00, 110.14s/it]

Source Res=384 Single: 5.892 ms/img | 169.7 FPS
Source Res=384 Batch=32: 1.785 ms/img | 560.4 FPS

====================================MaD-CoRN EXPERIMENT RESULTS=====================================

[1] TRAINING PERIODS RESULTS
                                               Training Periods Results                                              

+-------+------------+-----------+-----------+---------+---------+-----------+--------+--------+--------------------+
| Epoch | Train Loss | Train Acc | Train AUC | Val Acc | Val AUC | Precision | Recall | F1     | Uncertain Rate (%) |
+-------+------------+-----------+-----------+---------+---------+-----------+--------+--------+--------------------+
| 1     | 0.6956     | 0.4999    | 0.4990    | 0.1429  | 0.5096  | 0.0000    | 0.0000 | 0.0000 | 100.00             |
| 2     | 0.6846     | 0.5376    | 0.5496    | 0.3495  | 0.5958  | 0.9048    | 0.2694 | 0.4152 | 80.20              |
| 3     | 0.6701     | 0.5635    | 0.5746    | 0.3206  | 0.55

,Experiment Name,Dataset,Train Samples Original,Train Samples Used Balanced,Validation Samples,Test Samples,Train Real Used,Train Fake Used,Validation Real,Validation Fake,Test Real,Test Fake,Balancing Method,Resolutions Used,Input Modalities,RGB Shape,Device,CUDA Available,Framework,Torch Version
0,MaD-CoRN Balanced Train,FaceForensics++ C23,44756,76712,5600,5597,38356,38356,800,4800,800,4797,Train-only oversampling of real class with rep...,"[128, 224, 256, 384]",RGB,"(3, 224, 224)",cuda,True,PyTorch,2.11.0+cu130



Results saved to: ./mad_corn_results_ffpp_balanced


In [1]:
#FF++ with balance configuration
!pip install scikit-learn torch torchvision tqdm pandas pillow

import os, io, json, math, time, random, warnings
import numpy as np
import pandas as pd
from dataclasses import dataclass, asdict
from PIL import Image, ImageFile

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torchvision.transforms as T
from torch.cuda.amp import GradScaler, autocast

from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, accuracy_score

# ============================================================
# CONFIG
# ============================================================

CONFIG = dict(
    metadata_csv="/home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/metadata.csv",

    resolutions=[128, 224, 256, 384],
    image_size_for_model=224,

    epochs=15,
    batch_size=32,
    lr=1e-3,
    weight_decay=1e-4,
    label_smoothing=0.05,
    warmup_epochs=3,

    base_ch=32,

    num_workers=0,
    seed=42,
    save_dir="./mad_corn_results_ffpp_balanced",
    device="auto",
    smoke_test=False,
)

# ============================================================
# HELPERS
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def auto_device(requested):
    if requested != "auto":
        return torch.device(requested)
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

set_seed(CONFIG["seed"])
DEVICE = auto_device(CONFIG["device"])
os.makedirs(CONFIG["save_dir"], exist_ok=True)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", DEVICE)

# ============================================================
# RESULT CLASSES
# ============================================================

@dataclass
class EpochResult:
    epoch: int
    train_loss: float
    train_acc: float
    train_auc: float
    val_acc: float
    val_auc: float
    precision: float
    recall: float
    f1: float
    uncertain_rate: float
    lr: float

@dataclass
class TestResult:
    model_name: str
    accuracy: float
    auc: float
    precision: float
    recall: float
    f1: float
    uncertain_rate: float

@dataclass
class ResolutionResult:
    resolution: int
    accuracy: float
    auc: float
    fast_pct: float
    medium_pct: float
    full_pct: float

@dataclass
class InferenceResult:
    mode: str
    time_per_image_ms: float
    fps: float

@dataclass
class TrainingTimeResult:
    model_type: str
    minutes: float
    hours: float

# ============================================================
# MODEL
# ============================================================

class ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3, stride=1, padding=1, groups=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel, stride, padding, groups=groups, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU6(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class DepthwiseSeparable(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.dw = ConvBNReLU(in_ch, in_ch, kernel=3, stride=stride, padding=1, groups=in_ch)
        self.pw = ConvBNReLU(in_ch, out_ch, kernel=1, padding=0)

    def forward(self, x):
        return self.pw(self.dw(x))

class ConvReservoir(nn.Module):
    def __init__(self, in_ch, reservoir_ch=64, spectral_radius=0.9):
        super().__init__()
        self.input_proj = nn.Conv2d(in_ch, reservoir_ch, kernel_size=1, bias=False)
        self.input_bn = nn.BatchNorm2d(reservoir_ch)

        W = torch.randn(reservoir_ch, reservoir_ch, 3, 3)
        W = W / (W.abs().max() + 1e-8) * spectral_radius
        self.register_buffer("W_res", W)

        self.readout = nn.Sequential(
            nn.Conv2d(reservoir_ch, reservoir_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(reservoir_ch),
            nn.ReLU6(inplace=True),
        )

    def forward(self, x):
        h = F.relu(self.input_bn(self.input_proj(x)))
        h_res = torch.tanh(F.conv2d(h, self.W_res, padding=1))
        return self.readout(h + h_res)

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, mid),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        w = self.gate(x).view(x.size(0), x.size(1), 1, 1)
        return x * w

class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx, _ = x.max(dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))

class MultiScaleAttentionBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        b_ch = out_ch // 3

        self.branch1 = DepthwiseSeparable(in_ch, b_ch)
        self.branch2 = nn.Sequential(
            DepthwiseSeparable(in_ch, b_ch),
            DepthwiseSeparable(b_ch, b_ch),
        )
        self.branch3 = nn.Sequential(
            DepthwiseSeparable(in_ch, b_ch),
            DepthwiseSeparable(b_ch, b_ch),
            DepthwiseSeparable(b_ch, b_ch),
        )

        self.fuse = ConvBNReLU(b_ch * 3, out_ch, kernel=1, padding=0)
        self.ca = ChannelAttention(out_ch)
        self.sa = SpatialAttention()

    def forward(self, x):
        f = self.fuse(torch.cat([self.branch1(x), self.branch2(x), self.branch3(x)], dim=1))
        return self.sa(self.ca(f))

class MaDCoRN(nn.Module):
    def __init__(self, in_ch=3, num_classes=2, base_ch=32):
        super().__init__()
        c = base_ch

        self.stem = nn.Sequential(
            ConvBNReLU(in_ch, c, kernel=3, stride=2, padding=1),
            ConvBNReLU(c, c * 2, kernel=3, stride=1, padding=1),
        )

        self.mad1 = MultiScaleAttentionBlock(c * 2, c * 4)
        self.mad2 = MultiScaleAttentionBlock(c * 4, c * 4)
        self.res1 = ConvReservoir(c * 4, c * 4)
        self.down1 = nn.Conv2d(c * 4, c * 4, kernel_size=3, stride=2, padding=1, bias=False)

        self.mad3 = MultiScaleAttentionBlock(c * 4, c * 8)
        self.mad4 = MultiScaleAttentionBlock(c * 8, c * 8)
        self.res2 = ConvReservoir(c * 8, c * 8)
        self.down2 = nn.Conv2d(c * 8, c * 8, kernel_size=3, stride=2, padding=1, bias=False)

        self.mad5 = MultiScaleAttentionBlock(c * 8, c * 16)
        self.mad6 = MultiScaleAttentionBlock(c * 16, c * 16)
        self.res3 = ConvReservoir(c * 16, c * 16)

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(c * 16, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x)
        x = self.down1(self.res1(self.mad2(self.mad1(x))))
        x = self.down2(self.res2(self.mad4(self.mad3(x))))
        x = self.res3(self.mad6(self.mad5(x)))
        return self.head(self.pool(x))

    def count_parameters(self):
        total = sum(p.numel() for p in self.parameters())
        frozen = sum(p.numel() for p in self.parameters() if not p.requires_grad)
        return total, frozen

def freeze_reservoir_weights(model):
    for name, param in model.named_parameters():
        if "W_res" in name:
            param.requires_grad_(False)

# ============================================================
# DATASET
# ============================================================

class AddGaussianNoise:
    def __init__(self, std=0.01):
        self.std = std

    def __call__(self, t):
        return t + torch.randn_like(t) * self.std

class JPEGCompression:
    def __init__(self, quality_range=(60, 95)):
        self.lo, self.hi = quality_range

    def __call__(self, img):
        q = random.randint(self.lo, self.hi)
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=q)
        buf.seek(0)
        return Image.open(buf).copy()

_MEAN = [0.485, 0.456, 0.406]
_STD = [0.229, 0.224, 0.225]

def build_transforms(source_resolution, split="train", image_size_for_model=224):
    if split == "train":
        return T.Compose([
            T.Resize((source_resolution, source_resolution)),
            T.Resize((image_size_for_model + 16, image_size_for_model + 16)),
            T.RandomCrop(image_size_for_model),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.1),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
            T.RandomGrayscale(p=0.05),
            T.RandomApply([T.GaussianBlur(kernel_size=3)], p=0.15),
            JPEGCompression(quality_range=(60, 95)),
            T.ToTensor(),
            AddGaussianNoise(std=0.01),
            T.Normalize(_MEAN, _STD),
        ])

    return T.Compose([
        T.Resize((source_resolution, source_resolution)),
        T.Resize((image_size_for_model, image_size_for_model)),
        T.ToTensor(),
        T.Normalize(_MEAN, _STD),
    ])

def load_ffpp_dataframes(cfg):
    df = pd.read_csv(cfg["metadata_csv"])

    required = {"frame_path", "split", "label", "label_name", "video_id"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"metadata.csv missing columns: {missing}")

    df = df[df["frame_path"].apply(os.path.exists)].reset_index(drop=True)

    print("\nLoaded FF++ metadata")
    print("Total usable frames:", len(df))

    for split in ["Train", "Val", "Test"]:
        sdf = df[df["split"].str.lower() == split.lower()]
        real = int((sdf["label"] == 0).sum())
        fake = int((sdf["label"] == 1).sum())
        videos = sdf["video_id"].nunique()
        print(f"{split}: real={real}, fake={fake}, total={len(sdf)}, videos={videos}")

    train_full = df[df["split"].str.lower() == "train"].reset_index(drop=True)
    val_df = df[df["split"].str.lower() == "val"].reset_index(drop=True)
    test_df = df[df["split"].str.lower() == "test"].reset_index(drop=True)

    return train_full, val_df, test_df

def make_balanced_train_df(train_df, seed=42):
    real_df = train_df[train_df["label"] == 0]
    fake_df = train_df[train_df["label"] == 1]

    n = min(len(real_df), len(fake_df))

    real_bal = real_df.sample(n=n, random_state=seed)
    fake_bal = fake_df.sample(n=n, random_state=seed)

    balanced_df = (
        pd.concat([real_bal, fake_bal])
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )

    print("\nBalanced train set by undersampling")
    print(balanced_df["label_name"].value_counts())
    print("Total balanced train samples:", len(balanced_df))

    return balanced_df

def df_to_samples(df):
    return [(row["frame_path"], int(row["label"])) for _, row in df.iterrows()]

class MultiResolutionFFPPDataset(Dataset):
    def __init__(self, base_samples, resolutions, split="train", image_size_for_model=224):
        self.base_samples = list(base_samples)
        self.resolutions = list(resolutions)
        self.split = split
        self.image_size_for_model = image_size_for_model

        self.transforms = {
            res: build_transforms(res, split=split, image_size_for_model=image_size_for_model)
            for res in self.resolutions
        }

        if split == "train":
            self.samples = [(path, label, None) for path, label in self.base_samples]
        else:
            self.samples = [
                (path, label, res)
                for path, label in self.base_samples
                for res in self.resolutions
            ]

        n_real = sum(1 for _, label in self.base_samples if label == 0)
        n_fake = len(self.base_samples) - n_real

        print(
            f"[{split}] base: real={n_real:,}, fake={n_fake:,}, total={len(self.base_samples):,} | "
            f"effective={len(self.samples):,}"
        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label, resolution = self.samples[idx]

        if resolution is None:
            resolution = random.choice(self.resolutions)

        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            img = Image.fromarray(np.zeros((resolution, resolution, 3), dtype=np.uint8))

        x = self.transforms[resolution](img)
        y = torch.tensor(label, dtype=torch.long)

        return x, y

    def subset_for_resolution(self, resolution):
        return SingleResolutionView(
            self.base_samples,
            resolution,
            image_size_for_model=self.image_size_for_model,
        )

    @property
    def base_count(self):
        return len(self.base_samples)

    @property
    def expanded_count(self):
        return len(self.samples)

class SingleResolutionView(Dataset):
    def __init__(self, base_samples, resolution, image_size_for_model=224):
        self.base_samples = list(base_samples)
        self.resolution = resolution
        self.transform = build_transforms(
            resolution,
            split="val",
            image_size_for_model=image_size_for_model,
        )

    def __len__(self):
        return len(self.base_samples)

    def __getitem__(self, idx):
        path, label = self.base_samples[idx]

        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            img = Image.fromarray(np.zeros((self.resolution, self.resolution, 3), dtype=np.uint8))

        x = self.transform(img)
        y = torch.tensor(label, dtype=torch.long)

        return x, y

def _make_loader(dataset, batch_size, num_workers, shuffle, drop_last):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=False,
    )

def print_dataset_summary(split_datasets, resolutions):
    print("\nDataset size summary")
    for name in ["train", "val", "test"]:
        ds = split_datasets[name]
        print(
            f"{name.title():<5} | base={ds.base_count:,}, "
            f"effective={ds.expanded_count:,}, resolutions={resolutions}"
        )

    print("\nStrategy:")
    print("  Train: balanced undersampled train set + one random source resolution per image per epoch")
    print("  Val:   original val set + all requested source resolutions per image")
    print("  Test:  original test set + all requested source resolutions per image")
    print(f"  Model input tensor size: (3, {CONFIG['image_size_for_model']}, {CONFIG['image_size_for_model']})")

def make_dataloaders(cfg):
    train_full_df, val_df, test_df = load_ffpp_dataframes(cfg)
    train_balanced_df = make_balanced_train_df(train_full_df, seed=cfg["seed"])

    train_samples = df_to_samples(train_balanced_df)
    val_samples = df_to_samples(val_df)
    test_samples = df_to_samples(test_df)

    resolutions = list(cfg["resolutions"])

    split_datasets = {
        "train": MultiResolutionFFPPDataset(
            train_samples,
            resolutions,
            split="train",
            image_size_for_model=cfg["image_size_for_model"],
        ),
        "val": MultiResolutionFFPPDataset(
            val_samples,
            resolutions,
            split="val",
            image_size_for_model=cfg["image_size_for_model"],
        ),
        "test": MultiResolutionFFPPDataset(
            test_samples,
            resolutions,
            split="test",
            image_size_for_model=cfg["image_size_for_model"],
        ),
    }

    print_dataset_summary(split_datasets, resolutions)

    loaders = {
        "train": _make_loader(
            split_datasets["train"],
            cfg["batch_size"],
            cfg["num_workers"],
            shuffle=True,
            drop_last=True,
        ),
        "val": _make_loader(
            split_datasets["val"],
            cfg["batch_size"],
            cfg["num_workers"],
            shuffle=False,
            drop_last=False,
        ),
        "test": _make_loader(
            split_datasets["test"],
            cfg["batch_size"],
            cfg["num_workers"],
            shuffle=False,
            drop_last=False,
        ),
    }

    meta = {
        "train_full_df": train_full_df,
        "train_balanced_df": train_balanced_df,
        "val_df": val_df,
        "test_df": test_df,
    }

    return {
        "loaders": loaders,
        "datasets": split_datasets,
        "meta": meta,
    }

def make_smoke_loader(resolution, n=200, bs=8):
    imgs = torch.randn(n, 3, CONFIG["image_size_for_model"], CONFIG["image_size_for_model"])
    labels = torch.randint(0, 2, (n,))
    return DataLoader(TensorDataset(imgs, labels), batch_size=bs, shuffle=True)

# ============================================================
# TRAINING
# ============================================================

def compute_uncertain_rate(probs, threshold=0.65):
    conf = probs.max(axis=1)
    return float(np.mean(conf < threshold) * 100)

def run_epoch(model, loader, criterion, optimizer, device, scaler=None, is_train=True, desc=None):
    model.train(is_train)

    total_loss = 0.0
    seen = 0
    all_probs, all_preds, all_labels = [], [], []

    pbar = tqdm(
        loader,
        total=len(loader) if hasattr(loader, "__len__") else None,
        desc=desc or ("Train" if is_train else "Eval"),
        leave=False,
        dynamic_ncols=True,
        mininterval=0.5,
    )

    with torch.set_grad_enabled(is_train):
        for imgs, labels in pbar:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            if is_train:
                optimizer.zero_grad(set_to_none=True)

            if is_train and scaler:
                with autocast():
                    logits = model(imgs)
                    loss = criterion(logits, labels)

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()

            else:
                logits = model(imgs)
                loss = criterion(logits, labels)

                if is_train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()

            probs = torch.softmax(logits.float(), dim=1).detach().cpu().numpy()
            preds = probs.argmax(axis=1)

            all_probs.append(probs)
            all_preds.append(preds)
            all_labels.append(labels.detach().cpu().numpy())

            batch_n = len(labels)
            total_loss += loss.item() * batch_n
            seen += batch_n

            pbar.set_postfix(loss=f"{loss.item():.4f}", samples=f"{seen:,}")

    all_probs = np.vstack(all_probs)
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    acc = accuracy_score(all_labels, all_preds)

    try:
        auc = roc_auc_score(all_labels, all_probs[:, 1])
    except ValueError:
        auc = 0.5

    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    uncertain_rate = compute_uncertain_rate(all_probs)

    return {
        "loss": total_loss / max(seen, 1),
        "acc": acc,
        "auc": auc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "uncertain_rate": uncertain_rate,
        "probs": all_probs,
        "preds": all_preds,
        "labels": all_labels,
    }

class Trainer:
    def __init__(self, model, train_loader, val_loader, test_loader, device, cfg, ckpt_path):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader
        self.device = device
        self.epochs = cfg["epochs"]
        self.ckpt_path = ckpt_path
        self.model_name = "MaD-CoRN-FFPP-Balanced"

        self.criterion = nn.CrossEntropyLoss(label_smoothing=cfg["label_smoothing"])

        self.optimizer = optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=cfg["lr"],
            weight_decay=cfg["weight_decay"],
        )

        total_steps = self.epochs
        warmup_steps = cfg["warmup_epochs"]

        def lr_lambda(epoch):
            if epoch < warmup_steps:
                return max(1e-8, epoch / max(1, warmup_steps))
            prog = (epoch - warmup_steps) / max(1, total_steps - warmup_steps)
            return 0.5 * (1.0 + math.cos(math.pi * prog))

        self.scheduler = optim.lr_scheduler.LambdaLR(self.optimizer, lr_lambda)
        self.scaler = GradScaler() if device.type == "cuda" else None
        self.epoch_results = []

    def train(self):
        best_auc = -1.0
        t0 = time.time()

        for epoch in range(1, self.epochs + 1):
            tr = run_epoch(
                self.model,
                self.train_loader,
                self.criterion,
                self.optimizer,
                self.device,
                self.scaler,
                is_train=True,
                desc=f"Epoch {epoch}/{self.epochs} [train]",
            )

            with torch.no_grad():
                vl = run_epoch(
                    self.model,
                    self.val_loader,
                    self.criterion,
                    None,
                    self.device,
                    None,
                    is_train=False,
                    desc=f"Epoch {epoch}/{self.epochs} [val]",
                )

            self.scheduler.step()
            lr_now = self.optimizer.param_groups[0]["lr"]

            er = EpochResult(
                epoch=epoch,
                train_loss=tr["loss"],
                train_acc=tr["acc"],
                train_auc=tr["auc"],
                val_acc=vl["acc"],
                val_auc=vl["auc"],
                precision=vl["precision"],
                recall=vl["recall"],
                f1=vl["f1"],
                uncertain_rate=vl["uncertain_rate"],
                lr=lr_now,
            )

            self.epoch_results.append(er)

            print(
                f"Epoch {epoch:3d}/{self.epochs} | "
                f"Loss {er.train_loss:.4f} | "
                f"Tr Acc {er.train_acc:.4f} AUC {er.train_auc:.4f} | "
                f"Val Acc {er.val_acc:.4f} AUC {er.val_auc:.4f} | "
                f"P {er.precision:.4f} R {er.recall:.4f} F1 {er.f1:.4f} | "
                f"Unc {er.uncertain_rate:.1f}% | LR {lr_now:.2e}"
            )

            if vl["auc"] > best_auc:
                best_auc = vl["auc"]
                torch.save(self.model.state_dict(), self.ckpt_path)
                print(f"Checkpoint saved. Val AUC = {best_auc:.4f}")

        return self.epoch_results, (time.time() - t0) / 60

    def evaluate(self, loader=None, load_best=True):
        if load_best and os.path.exists(self.ckpt_path):
            self.model.load_state_dict(torch.load(self.ckpt_path, map_location=self.device))

        loader = loader or self.test_loader

        with torch.no_grad():
            r = run_epoch(
                self.model,
                loader,
                self.criterion,
                None,
                self.device,
                None,
                is_train=False,
                desc="Test evaluation",
            )

        return TestResult(
            model_name=self.model_name,
            accuracy=r["acc"],
            auc=r["auc"],
            precision=r["precision"],
            recall=r["recall"],
            f1=r["f1"],
            uncertain_rate=r["uncertain_rate"],
        )

# ============================================================
# EVAL / TIMING
# ============================================================

@torch.no_grad()
def measure_inference(model, resolution, device, image_size_for_model=224, batch_size=32, n_warmup=20, n_runs=200):
    model.eval()
    results = []

    for bs, label in [(1, "Single"), (batch_size, f"Batch={batch_size}")]:
        dummy = torch.randn(bs, 3, image_size_for_model, image_size_for_model, device=device)

        for _ in range(n_warmup):
            model(dummy)

        if device.type == "cuda":
            torch.cuda.synchronize()

        t0 = time.perf_counter()

        for _ in range(n_runs):
            model(dummy)

        if device.type == "cuda":
            torch.cuda.synchronize()

        elapsed = time.perf_counter() - t0
        total = bs * n_runs

        results.append(
            InferenceResult(
                mode=f"Source Res={resolution} {label}",
                time_per_image_ms=elapsed / total * 1000,
                fps=total / elapsed,
            )
        )

    return results

@torch.no_grad()
def evaluate_by_resolution(model, loader, device, resolution):
    model.eval()

    all_probs, all_labels = [], []
    buckets = {"fast": 0, "medium": 0, "full": 0}
    total = 0

    pbar = tqdm(
        loader,
        total=len(loader) if hasattr(loader, "__len__") else None,
        desc=f"Resolution {resolution} eval",
        leave=False,
        dynamic_ncols=True,
        mininterval=0.5,
    )

    for imgs, labels in pbar:
        imgs = imgs.to(device, non_blocking=True)
        bs = imgs.size(0)

        if device.type == "cuda":
            torch.cuda.synchronize()

        t0 = time.perf_counter()
        logits = model(imgs)

        if device.type == "cuda":
            torch.cuda.synchronize()

        ms = (time.perf_counter() - t0) / bs * 1000

        probs = torch.softmax(logits.float(), dim=1).cpu().numpy()

        all_probs.append(probs)
        all_labels.append(labels.numpy())

        if ms < 5:
            buckets["fast"] += bs
        elif ms < 20:
            buckets["medium"] += bs
        else:
            buckets["full"] += bs

        total += bs
        pbar.set_postfix(samples=f"{total:,}", ms_per_img=f"{ms:.3f}")

    all_probs = np.vstack(all_probs)
    all_labels = np.concatenate(all_labels)
    preds = all_probs.argmax(axis=1)

    acc = accuracy_score(all_labels, preds)

    try:
        auc = roc_auc_score(all_labels, all_probs[:, 1])
    except ValueError:
        auc = 0.5

    return ResolutionResult(
        resolution=resolution,
        accuracy=acc,
        auc=auc,
        fast_pct=buckets["fast"] / max(total, 1) * 100,
        medium_pct=buckets["medium"] / max(total, 1) * 100,
        full_pct=buckets["full"] / max(total, 1) * 100,
    )

# ============================================================
# REPORTING
# ============================================================

def _table(headers, rows, title=""):
    col_w = [
        max(len(str(h)), max((len(str(r[i])) for r in rows), default=0))
        for i, h in enumerate(headers)
    ]

    sep = "+" + "+".join("-" * (w + 2) for w in col_w) + "+"
    hdr = "|" + "|".join(f" {h:<{w}} " for h, w in zip(headers, col_w)) + "|"

    body = [
        "|" + "|".join(f" {str(v):<{w}} " for v, w in zip(r, col_w)) + "|"
        for r in rows
    ]

    out = [sep, hdr, sep] + body + [sep]

    if title:
        out.insert(0, title.center(len(sep)))
        out.insert(1, "")

    return "\n".join(out)

def print_training_table(ers):
    h = [
        "Epoch", "Train Loss", "Train Acc", "Train AUC",
        "Val Acc", "Val AUC", "Precision", "Recall", "F1",
        "Uncertain Rate (%)"
    ]

    rows = [
        [
            e.epoch,
            f"{e.train_loss:.4f}",
            f"{e.train_acc:.4f}",
            f"{e.train_auc:.4f}",
            f"{e.val_acc:.4f}",
            f"{e.val_auc:.4f}",
            f"{e.precision:.4f}",
            f"{e.recall:.4f}",
            f"{e.f1:.4f}",
            f"{e.uncertain_rate:.2f}",
        ]
        for e in ers
    ]

    t = _table(h, rows, "Training Periods Results")
    print(t)
    return t

def print_test_table(trs):
    h = ["Model", "Accuracy", "AUC", "Precision", "Recall", "F1", "Uncertain Rate (%)"]

    rows = [
        [
            r.model_name,
            f"{r.accuracy:.4f}",
            f"{r.auc:.4f}",
            f"{r.precision:.4f}",
            f"{r.recall:.4f}",
            f"{r.f1:.4f}",
            f"{r.uncertain_rate:.2f}",
        ]
        for r in trs
    ]

    t = _table(h, rows, "Test Results")
    print(t)
    return t

def print_resolution_table(rrs):
    h = ["Resolution", "Accuracy", "AUC", "Fast (%)", "Medium (%)", "Full (%)"]

    rows = [
        [
            r.resolution,
            f"{r.accuracy:.4f}",
            f"{r.auc:.4f}",
            f"{r.fast_pct:.2f}",
            f"{r.medium_pct:.2f}",
            f"{r.full_pct:.2f}",
        ]
        for r in rrs
    ]

    t = _table(h, rows, "Accuracy Based on Resolution")
    print(t)
    return t

def print_inference_table(irs):
    h = ["Mode", "Time per Image (ms)", "FPS"]

    rows = [
        [r.mode, f"{r.time_per_image_ms:.3f}", f"{r.fps:.1f}"]
        for r in irs
    ]

    t = _table(h, rows, "Inference Time")
    print(t)
    return t

def print_training_time_table(tts):
    h = ["Model Type", "min", "hrs"]

    rows = [
        [r.model_type, f"{r.minutes:.1f}", f"{r.hours:.2f}"]
        for r in tts
    ]

    t = _table(h, rows, "Training Time")
    print(t)
    return t

def save_all_results(epoch_results, test_results, res_results, inf_results, tt_results, save_dir, meta):
    os.makedirs(save_dir, exist_ok=True)

    data = {
        "epoch_results": [asdict(e) for e in epoch_results],
        "test_results": [asdict(t) for t in test_results],
        "res_results": [asdict(r) for r in res_results],
        "inference": [asdict(i) for i in inf_results],
        "training_time": [asdict(t) for t in tt_results],
    }

    with open(os.path.join(save_dir, "mad_corn_results.json"), "w") as f:
        json.dump(data, f, indent=2)

    for name, rows in data.items():
        if rows:
            pd.DataFrame(rows).to_csv(os.path.join(save_dir, f"mad_corn_{name}.csv"), index=False)

    train_full_df = meta["train_full_df"]
    train_balanced_df = meta["train_balanced_df"]
    val_df = meta["val_df"]
    test_df = meta["test_df"]

    experiment_summary = pd.DataFrame([{
        "Experiment Name": "MaD-CoRN Balanced Train",
        "Dataset": "FaceForensics++ C23",
        "Train Samples Original": len(train_full_df),
        "Train Samples Used Balanced": len(train_balanced_df),
        "Validation Samples": len(val_df),
        "Test Samples": len(test_df),
        "Train Real Used": int((train_balanced_df["label"] == 0).sum()),
        "Train Fake Used": int((train_balanced_df["label"] == 1).sum()),
        "Validation Real": int((val_df["label"] == 0).sum()),
        "Validation Fake": int((val_df["label"] == 1).sum()),
        "Test Real": int((test_df["label"] == 0).sum()),
        "Test Fake": int((test_df["label"] == 1).sum()),
        "Balancing Method": "Train-only undersampling of fake class to match real class",
        "Resolutions Used": str(CONFIG["resolutions"]),
        "Input Modalities": "RGB",
        "RGB Shape": f"(3, {CONFIG['image_size_for_model']}, {CONFIG['image_size_for_model']})",
        "Device": str(DEVICE),
        "CUDA Available": torch.cuda.is_available(),
        "Framework": "PyTorch",
        "Torch Version": torch.__version__,
        "Batching Strategy": "Train: balanced undersampled set with one random source resolution per image per epoch; Val/Test: original distribution with all requested source resolutions",
    }])

    experiment_summary.to_csv(os.path.join(save_dir, "experiment_summary.csv"), index=False)

    print("\nExperiment Summary")
    display(experiment_summary)

    print("\nResults saved to:", save_dir)

def print_full_report(epoch_results, test_results, res_results, inf_results, tt_results, save_dir, meta):
    div = "=" * 100

    print("\n" + div)
    print("MaD-CoRN EXPERIMENT RESULTS".center(100, "="))
    print(div)

    print("\n[1] TRAINING PERIODS RESULTS")
    print_training_table(epoch_results)

    print("\n[2] TEST RESULTS")
    print_test_table(test_results)

    print("\n[3] ACCURACY BASED ON RESOLUTION")
    print_resolution_table(res_results)

    print("\n[4] INFERENCE TIME")
    print_inference_table(inf_results)

    print("\n[5] TRAINING TIME")
    print_training_time_table(tt_results)

    save_all_results(epoch_results, test_results, res_results, inf_results, tt_results, save_dir, meta)

# ============================================================
# PIPELINE
# ============================================================

def run_pipeline(cfg):
    set_seed(cfg["seed"])
    device = auto_device(cfg["device"])
    os.makedirs(cfg["save_dir"], exist_ok=True)

    print("\n" + "=" * 70)
    print("MaD-CoRN Pipeline FF++ Balanced Train")
    print("Device:", device)
    print("Source resolutions:", cfg["resolutions"])
    print("Model input size:", cfg["image_size_for_model"])
    print("Epochs:", cfg["epochs"])
    print("Smoke test:", cfg["smoke_test"])
    print("=" * 70)

    model = MaDCoRN(in_ch=3, num_classes=2, base_ch=cfg["base_ch"])
    freeze_reservoir_weights(model)

    total, frozen = model.count_parameters()
    trainable = total - frozen
    print(f"\nModel params: {total:,} total | {trainable:,} trainable | {frozen:,} frozen")

    split_datasets = None
    meta = None

    if cfg["smoke_test"]:
        print("\nSMOKE TEST")
        bs = cfg["batch_size"]
        train_loader = make_smoke_loader(cfg["image_size_for_model"], n=320, bs=bs)
        val_loader = make_smoke_loader(cfg["image_size_for_model"], n=80, bs=bs)
        test_loader = make_smoke_loader(cfg["image_size_for_model"], n=80, bs=bs)
        cfg = {**cfg, "epochs": min(cfg["epochs"], 3)}
        meta = {
            "train_full_df": pd.DataFrame(),
            "train_balanced_df": pd.DataFrame(),
            "val_df": pd.DataFrame(),
            "test_df": pd.DataFrame(),
        }
    else:
        print("\nLoading FF++ dataset...")
        data_bundle = make_dataloaders(cfg)
        split_datasets = data_bundle["datasets"]
        train_loader = data_bundle["loaders"]["train"]
        val_loader = data_bundle["loaders"]["val"]
        test_loader = data_bundle["loaders"]["test"]
        meta = data_bundle["meta"]

    ckpt_path = os.path.join(cfg["save_dir"], "mad_corn_ffpp_balanced_multires.pt")

    print("\n" + "-" * 60)
    print("TRAINING")
    print("-" * 60)

    trainer = Trainer(
        model,
        train_loader,
        val_loader,
        test_loader,
        device,
        cfg,
        ckpt_path,
    )

    epoch_results, training_min = trainer.train()

    print(f"\nTraining done: {training_min:.1f} min ({training_min / 60:.2f} hrs)")

    print("\n" + "-" * 60)
    print("TEST EVALUATION ON COMBINED MULTI-RES TEST SET")
    print("-" * 60)

    test_result = trainer.evaluate(test_loader, load_best=True)
    test_results = [test_result]

    print("\n" + "-" * 60)
    print("PER-RESOLUTION BREAKDOWN")
    print("-" * 60)

    res_results = []
    inf_results = []

    for res in tqdm(cfg["resolutions"], desc="Resolution sweep", dynamic_ncols=True):
        print(f"\nResolution {res}")

        if cfg["smoke_test"]:
            res_loader = make_smoke_loader(cfg["image_size_for_model"], n=80, bs=cfg["batch_size"])
        else:
            res_ds = split_datasets["test"].subset_for_resolution(res)
            res_loader = DataLoader(
                res_ds,
                batch_size=cfg["batch_size"],
                shuffle=False,
                num_workers=cfg["num_workers"],
                pin_memory=(device.type == "cuda"),
                persistent_workers=False,
            )

        rr = evaluate_by_resolution(model, res_loader, device, res)
        res_results.append(rr)

        print(
            f"Acc={rr.accuracy:.4f} AUC={rr.auc:.4f} "
            f"Fast={rr.fast_pct:.2f}% Medium={rr.medium_pct:.2f}% Full={rr.full_pct:.2f}%"
        )

        infs = measure_inference(
            model,
            res,
            device,
            image_size_for_model=cfg["image_size_for_model"],
            batch_size=cfg["batch_size"],
        )
        inf_results.extend(infs)

        for ir in infs:
            print(f"{ir.mode}: {ir.time_per_image_ms:.3f} ms/img | {ir.fps:.1f} FPS")

    tt_results = [
        TrainingTimeResult(
            model_type="MaD-CoRN FF++ balanced train multi-resolution training",
            minutes=training_min,
            hours=training_min / 60,
        )
    ]

    print_full_report(
        epoch_results,
        test_results,
        res_results,
        inf_results,
        tt_results,
        cfg["save_dir"],
        meta,
    )

    return {
        "epoch_results": epoch_results,
        "test_results": test_results,
        "res_results": res_results,
        "inf_results": inf_results,
        "tt_results": tt_results,
        "model": model,
        "trainer": trainer,
        "split_datasets": split_datasets,
        "meta": meta,
    }

results = run_pipeline(CONFIG)

Torch: 2.11.0+cu130
CUDA available: True
Device: cuda

MaD-CoRN Pipeline FF++ Balanced Train
Device: cuda
Source resolutions: [128, 224, 256, 384]
Model input size: 224
Epochs: 15
Smoke test: False

Model params: 3,428,062 total | 3,428,062 trainable | 0 frozen

Loading FF++ dataset...

Loaded FF++ metadata
Total usable frames: 55953
Train: real=6400, fake=38356, total=44756, videos=5600
Val: real=800, fake=4800, total=5600, videos=700
Test: real=800, fake=4797, total=5597, videos=700

Balanced train set by undersampling
label_name
Real    6400
Fake    6400
Name: count, dtype: int64
Total balanced train samples: 12800
[train] base: real=6,400, fake=6,400, total=12,800 | effective=12,800
[val] base: real=800, fake=4,800, total=5,600 | effective=22,400
[test] base: real=800, fake=4,797, total=5,597 | effective=22,388

Dataset size summary
Train | base=12,800, effective=12,800, resolutions=[128, 224, 256, 384]
Val   | base=5,600, effective=22,400, resolutions=[128, 224, 256, 384]
Test  | 

Epoch   1/15 | Loss 0.6958 | Tr Acc 0.4998 AUC 0.4960 | Val Acc 0.1429 AUC 0.4994 | P 0.0000 R 0.0000 F1 0.0000 | Unc 100.0% | LR 3.33e-04
Checkpoint saved. Val AUC = 0.4994


Epoch   2/15 | Loss 0.6942 | Tr Acc 0.5063 AUC 0.5043 | Val Acc 0.2946 AUC 0.5095 | P 0.8779 R 0.2057 F1 0.3333 | Unc 97.8% | LR 6.67e-04
Checkpoint saved. Val AUC = 0.5095


Epoch   3/15 | Loss 0.6932 | Tr Acc 0.5187 AUC 0.5230 | Val Acc 0.1838 AUC 0.5000 | P 0.9935 R 0.0481 F1 0.0917 | Unc 100.0% | LR 1.00e-03


Epoch   4/15 | Loss 0.6940 | Tr Acc 0.5048 AUC 0.5091 | Val Acc 0.7200 AUC 0.5691 | P 0.8725 R 0.7885 F1 0.8284 | Unc 100.0% | LR 9.83e-04
Checkpoint saved. Val AUC = 0.5691


...

Acc=0.3039 AUC=0.5456 Fast=100.00% Medium=0.00% Full=0.00%


Resolution sweep: 100%|██████████| 4/4 [06:20<00:00, 95.23s/it]

Source Res=384 Single: 5.917 ms/img | 169.0 FPS
Source Res=384 Batch=32: 1.769 ms/img | 565.2 FPS

====================================MaD-CoRN EXPERIMENT RESULTS=====================================

[1] TRAINING PERIODS RESULTS
                                               Training Periods Results                                              

+-------+------------+-----------+-----------+---------+---------+-----------+--------+--------+--------------------+
| Epoch | Train Loss | Train Acc | Train AUC | Val Acc | Val AUC | Precision | Recall | F1     | Uncertain Rate (%) |
+-------+------------+-----------+-----------+---------+---------+-----------+--------+--------+--------------------+
| 1     | 0.6958     | 0.4998    | 0.4960    | 0.1429  | 0.4994  | 0.0000    | 0.0000 | 0.0000 | 100.00             |
| 2     | 0.6942     | 0.5063    | 0.5043    | 0.2946  | 0.5095  | 0.8779    | 0.2057 | 0.3333 | 97.81              |
| 3     | 0.6932     | 0.5187    | 0.5230    | 0.1838  | 0.50

,Experiment Name,Dataset,Train Samples Original,Train Samples Used Balanced,Validation Samples,Test Samples,Train Real Used,Train Fake Used,Validation Real,Validation Fake,...,Test Fake,Balancing Method,Resolutions Used,Input Modalities,RGB Shape,Device,CUDA Available,Framework,Torch Version,Batching Strategy
0,MaD-CoRN Balanced Train,FaceForensics++ C23,44756,12800,5600,5597,6400,6400,800,4800,...,4797,Train-only undersampling of fake class to matc...,"[128, 224, 256, 384]",RGB,"(3, 224, 224)",cuda,True,PyTorch,2.11.0+cu130,Train: balanced undersampled set with one rand...



Results saved to: ./mad_corn_results_ffpp_balanced


In [3]:
# !pip install scikit-learn torch torchvision tqdm pandas pillow

import os, io, json, math, time, random, warnings
import numpy as np
import pandas as pd
from dataclasses import dataclass, asdict
from PIL import Image, ImageFile

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torchvision.transforms as T
from torch.cuda.amp import GradScaler, autocast

from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, accuracy_score

# ============================================================
# CONFIG
# ============================================================

CONFIG = dict(
    metadata_csv="/home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/metadata.csv",
    frames_dir="/home/jovyan/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/ffpp_preprocessed/all_frames",

    resolutions=[128, 224, 256, 384],
    image_size_for_model=224,

    epochs=15,
    batch_size=32,
    lr=1e-3,
    weight_decay=1e-4,
    label_smoothing=0.05,
    warmup_epochs=3,

    base_ch=32,

    num_workers=0,
    seed=42,
    save_dir="./mad_corn_results_ffpp",
    device="auto",
    smoke_test=False,
)

# ============================================================
# HELPERS
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def auto_device(requested):
    if requested != "auto":
        return torch.device(requested)
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

set_seed(CONFIG["seed"])
DEVICE = auto_device(CONFIG["device"])
os.makedirs(CONFIG["save_dir"], exist_ok=True)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", DEVICE)

# ============================================================
# RESULT CLASSES
# ============================================================

@dataclass
class EpochResult:
    epoch: int
    train_loss: float
    train_acc: float
    train_auc: float
    val_acc: float
    val_auc: float
    precision: float
    recall: float
    f1: float
    uncertain_rate: float
    lr: float

@dataclass
class TestResult:
    model_name: str
    accuracy: float
    auc: float
    precision: float
    recall: float
    f1: float
    uncertain_rate: float

@dataclass
class ResolutionResult:
    resolution: int
    accuracy: float
    auc: float
    fast_pct: float
    medium_pct: float
    full_pct: float

@dataclass
class InferenceResult:
    mode: str
    time_per_image_ms: float
    fps: float

@dataclass
class TrainingTimeResult:
    model_type: str
    minutes: float
    hours: float

# ============================================================
# MODEL
# ============================================================

class ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3, stride=1, padding=1, groups=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel, stride, padding, groups=groups, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU6(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class DepthwiseSeparable(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.dw = ConvBNReLU(in_ch, in_ch, kernel=3, stride=stride, padding=1, groups=in_ch)
        self.pw = ConvBNReLU(in_ch, out_ch, kernel=1, padding=0)

    def forward(self, x):
        return self.pw(self.dw(x))

class ConvReservoir(nn.Module):
    def __init__(self, in_ch, reservoir_ch=64, spectral_radius=0.9):
        super().__init__()
        self.input_proj = nn.Conv2d(in_ch, reservoir_ch, kernel_size=1, bias=False)
        self.input_bn = nn.BatchNorm2d(reservoir_ch)

        W = torch.randn(reservoir_ch, reservoir_ch, 3, 3)
        W = W / (W.abs().max() + 1e-8) * spectral_radius
        self.register_buffer("W_res", W)

        self.readout = nn.Sequential(
            nn.Conv2d(reservoir_ch, reservoir_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(reservoir_ch),
            nn.ReLU6(inplace=True),
        )

    def forward(self, x):
        h = F.relu(self.input_bn(self.input_proj(x)))
        h_res = torch.tanh(F.conv2d(h, self.W_res, padding=1))
        return self.readout(h + h_res)

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, mid),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        w = self.gate(x).view(x.size(0), x.size(1), 1, 1)
        return x * w

class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx, _ = x.max(dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))

class MultiScaleAttentionBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        b_ch = out_ch // 3

        self.branch1 = DepthwiseSeparable(in_ch, b_ch)
        self.branch2 = nn.Sequential(
            DepthwiseSeparable(in_ch, b_ch),
            DepthwiseSeparable(b_ch, b_ch),
        )
        self.branch3 = nn.Sequential(
            DepthwiseSeparable(in_ch, b_ch),
            DepthwiseSeparable(b_ch, b_ch),
            DepthwiseSeparable(b_ch, b_ch),
        )

        self.fuse = ConvBNReLU(b_ch * 3, out_ch, kernel=1, padding=0)
        self.ca = ChannelAttention(out_ch)
        self.sa = SpatialAttention()

    def forward(self, x):
        f = self.fuse(torch.cat([self.branch1(x), self.branch2(x), self.branch3(x)], dim=1))
        return self.sa(self.ca(f))

class MaDCoRN(nn.Module):
    def __init__(self, in_ch=3, num_classes=2, base_ch=32):
        super().__init__()
        c = base_ch

        self.stem = nn.Sequential(
            ConvBNReLU(in_ch, c, kernel=3, stride=2, padding=1),
            ConvBNReLU(c, c * 2, kernel=3, stride=1, padding=1),
        )

        self.mad1 = MultiScaleAttentionBlock(c * 2, c * 4)
        self.mad2 = MultiScaleAttentionBlock(c * 4, c * 4)
        self.res1 = ConvReservoir(c * 4, c * 4)
        self.down1 = nn.Conv2d(c * 4, c * 4, kernel_size=3, stride=2, padding=1, bias=False)

        self.mad3 = MultiScaleAttentionBlock(c * 4, c * 8)
        self.mad4 = MultiScaleAttentionBlock(c * 8, c * 8)
        self.res2 = ConvReservoir(c * 8, c * 8)
        self.down2 = nn.Conv2d(c * 8, c * 8, kernel_size=3, stride=2, padding=1, bias=False)

        self.mad5 = MultiScaleAttentionBlock(c * 8, c * 16)
        self.mad6 = MultiScaleAttentionBlock(c * 16, c * 16)
        self.res3 = ConvReservoir(c * 16, c * 16)

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(c * 16, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x)
        x = self.down1(self.res1(self.mad2(self.mad1(x))))
        x = self.down2(self.res2(self.mad4(self.mad3(x))))
        x = self.res3(self.mad6(self.mad5(x)))
        return self.head(self.pool(x))

    def count_parameters(self):
        total = sum(p.numel() for p in self.parameters())
        frozen = sum(p.numel() for p in self.parameters() if not p.requires_grad)
        return total, frozen

def freeze_reservoir_weights(model):
    for name, param in model.named_parameters():
        if "W_res" in name:
            param.requires_grad_(False)

# ============================================================
# DATASET
# ============================================================

class AddGaussianNoise:
    def __init__(self, std=0.01):
        self.std = std

    def __call__(self, t):
        return t + torch.randn_like(t) * self.std

class JPEGCompression:
    def __init__(self, quality_range=(60, 95)):
        self.lo, self.hi = quality_range

    def __call__(self, img):
        q = random.randint(self.lo, self.hi)
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=q)
        buf.seek(0)
        return Image.open(buf).copy()

_MEAN = [0.485, 0.456, 0.406]
_STD = [0.229, 0.224, 0.225]

def build_transforms(source_resolution, split="train", image_size_for_model=224):
    """
    Source resolution simulates compression/downsampling.
    Final tensor size is always image_size_for_model.
    This prevents DataLoader stack errors.
    """
    if split == "train":
        return T.Compose([
            T.Resize((source_resolution, source_resolution)),
            T.Resize((image_size_for_model + 16, image_size_for_model + 16)),
            T.RandomCrop(image_size_for_model),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.1),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
            T.RandomGrayscale(p=0.05),
            T.RandomApply([T.GaussianBlur(kernel_size=3)], p=0.15),
            JPEGCompression(quality_range=(60, 95)),
            T.ToTensor(),
            AddGaussianNoise(std=0.01),
            T.Normalize(_MEAN, _STD),
        ])

    return T.Compose([
        T.Resize((source_resolution, source_resolution)),
        T.Resize((image_size_for_model, image_size_for_model)),
        T.ToTensor(),
        T.Normalize(_MEAN, _STD),
    ])

def load_ffpp_splits(cfg):
    df = pd.read_csv(cfg["metadata_csv"])

    required = {"frame_path", "split", "label", "label_name", "video_id"}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(f"metadata.csv missing columns: {missing}")

    df = df[df["frame_path"].apply(os.path.exists)].reset_index(drop=True)

    print("\nLoaded FF++ metadata")
    print("Total usable frames:", len(df))

    split_map = {}

    for split_name, key in [("Train", "train"), ("Val", "val"), ("Test", "test")]:
        sdf = df[df["split"].str.lower() == split_name.lower()].reset_index(drop=True)
        samples = [(row["frame_path"], int(row["label"])) for _, row in sdf.iterrows()]
        split_map[key] = samples

        real = int((sdf["label"] == 0).sum())
        fake = int((sdf["label"] == 1).sum())
        videos = sdf["video_id"].nunique()

        print(f"{split_name}: real={real}, fake={fake}, total={len(sdf)}, videos={videos}")

    return split_map

class MultiResolutionFFPPDataset(Dataset):
    def __init__(self, base_samples, resolutions, split="train", image_size_for_model=224):
        self.base_samples = list(base_samples)
        self.resolutions = list(resolutions)
        self.split = split
        self.image_size_for_model = image_size_for_model

        self.transforms = {
            res: build_transforms(res, split=split, image_size_for_model=image_size_for_model)
            for res in self.resolutions
        }

        if split == "train":
            self.samples = [(path, label, None) for path, label in self.base_samples]
        else:
            self.samples = [
                (path, label, res)
                for path, label in self.base_samples
                for res in self.resolutions
            ]

        n_real = sum(1 for _, label in self.base_samples if label == 0)
        n_fake = len(self.base_samples) - n_real

        print(
            f"[{split}] base: real={n_real:,}, fake={n_fake:,}, total={len(self.base_samples):,} | "
            f"effective={len(self.samples):,}"
        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label, resolution = self.samples[idx]

        if resolution is None:
            resolution = random.choice(self.resolutions)

        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            img = Image.fromarray(np.zeros((resolution, resolution, 3), dtype=np.uint8))

        x = self.transforms[resolution](img)
        y = torch.tensor(label, dtype=torch.long)

        return x, y

    def subset_for_resolution(self, resolution):
        return SingleResolutionView(
            self.base_samples,
            resolution,
            image_size_for_model=self.image_size_for_model,
        )

    @property
    def base_count(self):
        return len(self.base_samples)

    @property
    def expanded_count(self):
        return len(self.samples)

class SingleResolutionView(Dataset):
    def __init__(self, base_samples, resolution, image_size_for_model=224):
        self.base_samples = list(base_samples)
        self.resolution = resolution
        self.transform = build_transforms(
            resolution,
            split="val",
            image_size_for_model=image_size_for_model,
        )

    def __len__(self):
        return len(self.base_samples)

    def __getitem__(self, idx):
        path, label = self.base_samples[idx]

        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            img = Image.fromarray(np.zeros((self.resolution, self.resolution, 3), dtype=np.uint8))

        x = self.transform(img)
        y = torch.tensor(label, dtype=torch.long)

        return x, y

def _make_loader(dataset, batch_size, num_workers, shuffle, drop_last):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=False,
    )

def print_dataset_summary(split_datasets, resolutions):
    print("\nDataset size summary")
    for name in ["train", "val", "test"]:
        ds = split_datasets[name]
        print(
            f"{name.title():<5} | base={ds.base_count:,}, "
            f"effective={ds.expanded_count:,}, resolutions={resolutions}"
        )

    print("\nStrategy:")
    print("  Train: one random source resolution per image per epoch")
    print("  Val:   all requested source resolutions per image")
    print("  Test:  all requested source resolutions per image")
    print(f"  Model input tensor size: (3, {CONFIG['image_size_for_model']}, {CONFIG['image_size_for_model']})")

def make_dataloaders(cfg):
    split_map = load_ffpp_splits(cfg)
    resolutions = list(cfg["resolutions"])

    split_datasets = {
        "train": MultiResolutionFFPPDataset(
            split_map["train"],
            resolutions,
            split="train",
            image_size_for_model=cfg["image_size_for_model"],
        ),
        "val": MultiResolutionFFPPDataset(
            split_map["val"],
            resolutions,
            split="val",
            image_size_for_model=cfg["image_size_for_model"],
        ),
        "test": MultiResolutionFFPPDataset(
            split_map["test"],
            resolutions,
            split="test",
            image_size_for_model=cfg["image_size_for_model"],
        ),
    }

    print_dataset_summary(split_datasets, resolutions)

    loaders = {
        "train": _make_loader(
            split_datasets["train"],
            cfg["batch_size"],
            cfg["num_workers"],
            shuffle=True,
            drop_last=True,
        ),
        "val": _make_loader(
            split_datasets["val"],
            cfg["batch_size"],
            cfg["num_workers"],
            shuffle=False,
            drop_last=False,
        ),
        "test": _make_loader(
            split_datasets["test"],
            cfg["batch_size"],
            cfg["num_workers"],
            shuffle=False,
            drop_last=False,
        ),
    }

    return {
        "loaders": loaders,
        "datasets": split_datasets,
        "base_splits": split_map,
    }

def make_smoke_loader(resolution, n=200, bs=8):
    imgs = torch.randn(n, 3, CONFIG["image_size_for_model"], CONFIG["image_size_for_model"])
    labels = torch.randint(0, 2, (n,))
    return DataLoader(TensorDataset(imgs, labels), batch_size=bs, shuffle=True)

# ============================================================
# TRAINING
# ============================================================

def compute_uncertain_rate(probs, threshold=0.65):
    conf = probs.max(axis=1)
    return float(np.mean(conf < threshold) * 100)

def run_epoch(model, loader, criterion, optimizer, device, scaler=None, is_train=True, desc=None):
    model.train(is_train)

    total_loss = 0.0
    seen = 0
    all_probs, all_preds, all_labels = [], [], []

    pbar = tqdm(
        loader,
        total=len(loader) if hasattr(loader, "__len__") else None,
        desc=desc or ("Train" if is_train else "Eval"),
        leave=False,
        dynamic_ncols=True,
        mininterval=0.5,
    )

    with torch.set_grad_enabled(is_train):
        for imgs, labels in pbar:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            if is_train:
                optimizer.zero_grad(set_to_none=True)

            if is_train and scaler:
                with autocast():
                    logits = model(imgs)
                    loss = criterion(logits, labels)

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()

            else:
                logits = model(imgs)
                loss = criterion(logits, labels)

                if is_train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()

            probs = torch.softmax(logits.float(), dim=1).detach().cpu().numpy()
            preds = probs.argmax(axis=1)

            all_probs.append(probs)
            all_preds.append(preds)
            all_labels.append(labels.detach().cpu().numpy())

            batch_n = len(labels)
            total_loss += loss.item() * batch_n
            seen += batch_n

            pbar.set_postfix(loss=f"{loss.item():.4f}", samples=f"{seen:,}")

    all_probs = np.vstack(all_probs)
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    acc = accuracy_score(all_labels, all_preds)

    try:
        auc = roc_auc_score(all_labels, all_probs[:, 1])
    except ValueError:
        auc = 0.5

    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    uncertain_rate = compute_uncertain_rate(all_probs)

    return {
        "loss": total_loss / max(seen, 1),
        "acc": acc,
        "auc": auc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "uncertain_rate": uncertain_rate,
        "probs": all_probs,
        "preds": all_preds,
        "labels": all_labels,
    }

class Trainer:
    def __init__(self, model, train_loader, val_loader, test_loader, device, cfg, ckpt_path):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader
        self.device = device
        self.epochs = cfg["epochs"]
        self.ckpt_path = ckpt_path
        self.model_name = "MaD-CoRN-FFPP"

        self.criterion = nn.CrossEntropyLoss(label_smoothing=cfg["label_smoothing"])

        self.optimizer = optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=cfg["lr"],
            weight_decay=cfg["weight_decay"],
        )

        total_steps = self.epochs
        warmup_steps = cfg["warmup_epochs"]

        def lr_lambda(epoch):
            if epoch < warmup_steps:
                return max(1e-8, epoch / max(1, warmup_steps))
            prog = (epoch - warmup_steps) / max(1, total_steps - warmup_steps)
            return 0.5 * (1.0 + math.cos(math.pi * prog))

        self.scheduler = optim.lr_scheduler.LambdaLR(self.optimizer, lr_lambda)
        self.scaler = GradScaler() if device.type == "cuda" else None
        self.epoch_results = []

    def train(self):
        best_auc = -1.0
        t0 = time.time()

        for epoch in range(1, self.epochs + 1):
            tr = run_epoch(
                self.model,
                self.train_loader,
                self.criterion,
                self.optimizer,
                self.device,
                self.scaler,
                is_train=True,
                desc=f"Epoch {epoch}/{self.epochs} [train]",
            )

            with torch.no_grad():
                vl = run_epoch(
                    self.model,
                    self.val_loader,
                    self.criterion,
                    None,
                    self.device,
                    None,
                    is_train=False,
                    desc=f"Epoch {epoch}/{self.epochs} [val]",
                )

            self.scheduler.step()
            lr_now = self.optimizer.param_groups[0]["lr"]

            er = EpochResult(
                epoch=epoch,
                train_loss=tr["loss"],
                train_acc=tr["acc"],
                train_auc=tr["auc"],
                val_acc=vl["acc"],
                val_auc=vl["auc"],
                precision=vl["precision"],
                recall=vl["recall"],
                f1=vl["f1"],
                uncertain_rate=vl["uncertain_rate"],
                lr=lr_now,
            )

            self.epoch_results.append(er)

            print(
                f"Epoch {epoch:3d}/{self.epochs} | "
                f"Loss {er.train_loss:.4f} | "
                f"Tr Acc {er.train_acc:.4f} AUC {er.train_auc:.4f} | "
                f"Val Acc {er.val_acc:.4f} AUC {er.val_auc:.4f} | "
                f"P {er.precision:.4f} R {er.recall:.4f} F1 {er.f1:.4f} | "
                f"Unc {er.uncertain_rate:.1f}% | LR {lr_now:.2e}"
            )

            if vl["auc"] > best_auc:
                best_auc = vl["auc"]
                torch.save(self.model.state_dict(), self.ckpt_path)
                print(f"Checkpoint saved. Val AUC = {best_auc:.4f}")

        return self.epoch_results, (time.time() - t0) / 60

    def evaluate(self, loader=None, load_best=True):
        if load_best and os.path.exists(self.ckpt_path):
            self.model.load_state_dict(torch.load(self.ckpt_path, map_location=self.device))

        loader = loader or self.test_loader

        with torch.no_grad():
            r = run_epoch(
                self.model,
                loader,
                self.criterion,
                None,
                self.device,
                None,
                is_train=False,
                desc="Test evaluation",
            )

        return TestResult(
            model_name=self.model_name,
            accuracy=r["acc"],
            auc=r["auc"],
            precision=r["precision"],
            recall=r["recall"],
            f1=r["f1"],
            uncertain_rate=r["uncertain_rate"],
        )

# ============================================================
# EVAL / TIMING
# ============================================================

@torch.no_grad()
def measure_inference(model, resolution, device, image_size_for_model=224, batch_size=32, n_warmup=20, n_runs=200):
    model.eval()
    results = []

    for bs, label in [(1, "Single"), (batch_size, f"Batch={batch_size}")]:
        dummy = torch.randn(bs, 3, image_size_for_model, image_size_for_model, device=device)

        for _ in range(n_warmup):
            model(dummy)

        if device.type == "cuda":
            torch.cuda.synchronize()

        t0 = time.perf_counter()

        for _ in range(n_runs):
            model(dummy)

        if device.type == "cuda":
            torch.cuda.synchronize()

        elapsed = time.perf_counter() - t0
        total = bs * n_runs

        results.append(
            InferenceResult(
                mode=f"Source Res={resolution} {label}",
                time_per_image_ms=elapsed / total * 1000,
                fps=total / elapsed,
            )
        )

    return results

@torch.no_grad()
def evaluate_by_resolution(model, loader, device, resolution):
    model.eval()

    all_probs, all_labels = [], []
    buckets = {"fast": 0, "medium": 0, "full": 0}
    total = 0

    pbar = tqdm(
        loader,
        total=len(loader) if hasattr(loader, "__len__") else None,
        desc=f"Resolution {resolution} eval",
        leave=False,
        dynamic_ncols=True,
        mininterval=0.5,
    )

    for imgs, labels in pbar:
        imgs = imgs.to(device, non_blocking=True)
        bs = imgs.size(0)

        if device.type == "cuda":
            torch.cuda.synchronize()

        t0 = time.perf_counter()
        logits = model(imgs)

        if device.type == "cuda":
            torch.cuda.synchronize()

        ms = (time.perf_counter() - t0) / bs * 1000

        probs = torch.softmax(logits.float(), dim=1).cpu().numpy()

        all_probs.append(probs)
        all_labels.append(labels.numpy())

        if ms < 5:
            buckets["fast"] += bs
        elif ms < 20:
            buckets["medium"] += bs
        else:
            buckets["full"] += bs

        total += bs
        pbar.set_postfix(samples=f"{total:,}", ms_per_img=f"{ms:.3f}")

    all_probs = np.vstack(all_probs)
    all_labels = np.concatenate(all_labels)
    preds = all_probs.argmax(axis=1)

    acc = accuracy_score(all_labels, preds)

    try:
        auc = roc_auc_score(all_labels, all_probs[:, 1])
    except ValueError:
        auc = 0.5

    return ResolutionResult(
        resolution=resolution,
        accuracy=acc,
        auc=auc,
        fast_pct=buckets["fast"] / max(total, 1) * 100,
        medium_pct=buckets["medium"] / max(total, 1) * 100,
        full_pct=buckets["full"] / max(total, 1) * 100,
    )

# ============================================================
# REPORTING
# ============================================================

def _table(headers, rows, title=""):
    col_w = [
        max(len(str(h)), max((len(str(r[i])) for r in rows), default=0))
        for i, h in enumerate(headers)
    ]

    sep = "+" + "+".join("-" * (w + 2) for w in col_w) + "+"
    hdr = "|" + "|".join(f" {h:<{w}} " for h, w in zip(headers, col_w)) + "|"

    body = [
        "|" + "|".join(f" {str(v):<{w}} " for v, w in zip(r, col_w)) + "|"
        for r in rows
    ]

    out = [sep, hdr, sep] + body + [sep]

    if title:
        out.insert(0, title.center(len(sep)))
        out.insert(1, "")

    return "\n".join(out)

def print_training_table(ers):
    h = [
        "Epoch", "Train Loss", "Train Acc", "Train AUC",
        "Val Acc", "Val AUC", "Precision", "Recall", "F1",
        "Uncertain Rate (%)"
    ]

    rows = [
        [
            e.epoch,
            f"{e.train_loss:.4f}",
            f"{e.train_acc:.4f}",
            f"{e.train_auc:.4f}",
            f"{e.val_acc:.4f}",
            f"{e.val_auc:.4f}",
            f"{e.precision:.4f}",
            f"{e.recall:.4f}",
            f"{e.f1:.4f}",
            f"{e.uncertain_rate:.2f}",
        ]
        for e in ers
    ]

    t = _table(h, rows, "Training Periods Results")
    print(t)
    return t

def print_test_table(trs):
    h = ["Model", "Accuracy", "AUC", "Precision", "Recall", "F1", "Uncertain Rate (%)"]

    rows = [
        [
            r.model_name,
            f"{r.accuracy:.4f}",
            f"{r.auc:.4f}",
            f"{r.precision:.4f}",
            f"{r.recall:.4f}",
            f"{r.f1:.4f}",
            f"{r.uncertain_rate:.2f}",
        ]
        for r in trs
    ]

    t = _table(h, rows, "Test Results")
    print(t)
    return t

def print_resolution_table(rrs):
    h = ["Resolution", "Accuracy", "AUC", "Fast (%)", "Medium (%)", "Full (%)"]

    rows = [
        [
            r.resolution,
            f"{r.accuracy:.4f}",
            f"{r.auc:.4f}",
            f"{r.fast_pct:.2f}",
            f"{r.medium_pct:.2f}",
            f"{r.full_pct:.2f}",
        ]
        for r in rrs
    ]

    t = _table(h, rows, "Accuracy Based on Resolution")
    print(t)
    return t

def print_inference_table(irs):
    h = ["Mode", "Time per Image (ms)", "FPS"]

    rows = [
        [r.mode, f"{r.time_per_image_ms:.3f}", f"{r.fps:.1f}"]
        for r in irs
    ]

    t = _table(h, rows, "Inference Time")
    print(t)
    return t

def print_training_time_table(tts):
    h = ["Model Type", "min", "hrs"]

    rows = [
        [r.model_type, f"{r.minutes:.1f}", f"{r.hours:.2f}"]
        for r in tts
    ]

    t = _table(h, rows, "Training Time")
    print(t)
    return t

def save_all_results(epoch_results, test_results, res_results, inf_results, tt_results, save_dir):
    os.makedirs(save_dir, exist_ok=True)

    data = {
        "epoch_results": [asdict(e) for e in epoch_results],
        "test_results": [asdict(t) for t in test_results],
        "res_results": [asdict(r) for r in res_results],
        "inference": [asdict(i) for i in inf_results],
        "training_time": [asdict(t) for t in tt_results],
    }

    with open(os.path.join(save_dir, "mad_corn_results.json"), "w") as f:
        json.dump(data, f, indent=2)

    for name, rows in data.items():
        if rows:
            pd.DataFrame(rows).to_csv(os.path.join(save_dir, f"mad_corn_{name}.csv"), index=False)

    print("\nResults saved to:", save_dir)

def print_full_report(epoch_results, test_results, res_results, inf_results, tt_results, save_dir):
    div = "=" * 100

    print("\n" + div)
    print("MaD-CoRN EXPERIMENT RESULTS".center(100, "="))
    print(div)

    print("\n[1] TRAINING PERIODS RESULTS")
    print_training_table(epoch_results)

    print("\n[2] TEST RESULTS")
    print_test_table(test_results)

    print("\n[3] ACCURACY BASED ON RESOLUTION")
    print_resolution_table(res_results)

    print("\n[4] INFERENCE TIME")
    print_inference_table(inf_results)

    print("\n[5] TRAINING TIME")
    print_training_time_table(tt_results)

    save_all_results(epoch_results, test_results, res_results, inf_results, tt_results, save_dir)

# ============================================================
# PIPELINE
# ============================================================

def run_pipeline(cfg):
    set_seed(cfg["seed"])
    device = auto_device(cfg["device"])
    os.makedirs(cfg["save_dir"], exist_ok=True)

    print("\n" + "=" * 70)
    print("MaD-CoRN Pipeline FF++")
    print("Device:", device)
    print("Source resolutions:", cfg["resolutions"])
    print("Model input size:", cfg["image_size_for_model"])
    print("Epochs:", cfg["epochs"])
    print("Smoke test:", cfg["smoke_test"])
    print("=" * 70)

    model = MaDCoRN(in_ch=3, num_classes=2, base_ch=cfg["base_ch"])
    freeze_reservoir_weights(model)

    total, frozen = model.count_parameters()
    print(f"\nModel params: {total:,} total | {frozen:,} frozen")

    split_datasets = None

    if cfg["smoke_test"]:
        print("\nSMOKE TEST")
        bs = cfg["batch_size"]
        train_loader = make_smoke_loader(cfg["image_size_for_model"], n=320, bs=bs)
        val_loader = make_smoke_loader(cfg["image_size_for_model"], n=80, bs=bs)
        test_loader = make_smoke_loader(cfg["image_size_for_model"], n=80, bs=bs)
        cfg = {**cfg, "epochs": min(cfg["epochs"], 3)}
    else:
        print("\nLoading FF++ dataset...")
        data_bundle = make_dataloaders(cfg)
        split_datasets = data_bundle["datasets"]
        train_loader = data_bundle["loaders"]["train"]
        val_loader = data_bundle["loaders"]["val"]
        test_loader = data_bundle["loaders"]["test"]

    ckpt_path = os.path.join(cfg["save_dir"], "mad_corn_ffpp_multires.pt")

    print("\n" + "-" * 60)
    print("TRAINING")
    print("-" * 60)

    trainer = Trainer(
        model,
        train_loader,
        val_loader,
        test_loader,
        device,
        cfg,
        ckpt_path,
    )

    epoch_results, training_min = trainer.train()

    print(f"\nTraining done: {training_min:.1f} min ({training_min / 60:.2f} hrs)")

    print("\n" + "-" * 60)
    print("TEST EVALUATION ON COMBINED MULTI-RES TEST SET")
    print("-" * 60)

    test_result = trainer.evaluate(test_loader, load_best=True)
    test_results = [test_result]

    print("\n" + "-" * 60)
    print("PER-RESOLUTION BREAKDOWN")
    print("-" * 60)

    res_results = []
    inf_results = []

    for res in tqdm(cfg["resolutions"], desc="Resolution sweep", dynamic_ncols=True):
        print(f"\nResolution {res}")

        if cfg["smoke_test"]:
            res_loader = make_smoke_loader(cfg["image_size_for_model"], n=80, bs=cfg["batch_size"])
        else:
            res_ds = split_datasets["test"].subset_for_resolution(res)
            res_loader = DataLoader(
                res_ds,
                batch_size=cfg["batch_size"],
                shuffle=False,
                num_workers=cfg["num_workers"],
                pin_memory=(device.type == "cuda"),
                persistent_workers=False,
            )

        rr = evaluate_by_resolution(model, res_loader, device, res)
        res_results.append(rr)

        print(
            f"Acc={rr.accuracy:.4f} AUC={rr.auc:.4f} "
            f"Fast={rr.fast_pct:.2f}% Medium={rr.medium_pct:.2f}% Full={rr.full_pct:.2f}%"
        )

        infs = measure_inference(
            model,
            res,
            device,
            image_size_for_model=cfg["image_size_for_model"],
            batch_size=cfg["batch_size"],
        )
        inf_results.extend(infs)

        for ir in infs:
            print(f"{ir.mode}: {ir.time_per_image_ms:.3f} ms/img | {ir.fps:.1f} FPS")

    tt_results = [
        TrainingTimeResult(
            model_type="MaD-CoRN FF++ multi-resolution training",
            minutes=training_min,
            hours=training_min / 60,
        )
    ]

    print_full_report(
        epoch_results,
        test_results,
        res_results,
        inf_results,
        tt_results,
        cfg["save_dir"],
    )

    return {
        "epoch_results": epoch_results,
        "test_results": test_results,
        "res_results": res_results,
        "inf_results": inf_results,
        "tt_results": tt_results,
        "model": model,
        "trainer": trainer,
        "split_datasets": split_datasets,
    }

results = run_pipeline(CONFIG)

Torch: 2.11.0+cu130
CUDA available: True
Device: cuda

MaD-CoRN Pipeline FF++
Device: cuda
Source resolutions: [128, 224, 256, 384]
Model input size: 224
Epochs: 15
Smoke test: False

Model params: 3,428,062 total | 0 frozen

Loading FF++ dataset...

Loaded FF++ metadata
Total usable frames: 55953
Train: real=6400, fake=38356, total=44756, videos=5600
Val: real=800, fake=4800, total=5600, videos=700
Test: real=800, fake=4797, total=5597, videos=700
[train] base: real=6,400, fake=38,356, total=44,756 | effective=44,756
[val] base: real=800, fake=4,800, total=5,600 | effective=22,400
[test] base: real=800, fake=4,797, total=5,597 | effective=22,388

Dataset size summary
Train | base=44,756, effective=44,756, resolutions=[128, 224, 256, 384]
Val   | base=5,600, effective=22,400, resolutions=[128, 224, 256, 384]
Test  | base=5,597, effective=22,388, resolutions=[128, 224, 256, 384]

Strategy:
  Train: one random source resolution per image per epoch
  Val:   all requested source resolution

Epoch   1/15 | Loss 0.7403 | Tr Acc 0.1444 AUC 0.4939 | Val Acc 0.1429 AUC 0.4981 | P 0.0000 R 0.0000 F1 0.0000 | Unc 100.0% | LR 3.33e-04
Checkpoint saved. Val AUC = 0.4981


Epoch   2/15 | Loss 0.4496 | Tr Acc 0.8565 AUC 0.5112 | Val Acc 0.8571 AUC 0.5343 | P 0.8571 R 1.0000 F1 0.9231 | Unc 0.0% | LR 6.67e-04
Checkpoint saved. Val AUC = 0.5343


Epoch   3/15 | Loss 0.4514 | Tr Acc 0.8570 AUC 0.5214 | Val Acc 0.8571 AUC 0.5348 | P 0.8571 R 1.0000 F1 0.9231 | Unc 0.0% | LR 1.00e-03
Checkpoint saved. Val AUC = 0.5348


Epoch   5/15 | Loss 0.4404 | Tr Acc 0.8570 AUC 0.5411 | Val Acc 0.8571 AUC 0.5376 | P 0.8571 R 1.0000 F1 0.9231 | Unc 0.0% | LR 9.33e-04
Checkpoint saved. Val AUC = 0.5376


...

Source Res=256 Single: 5.846 ms/img | 171.1 FPS
Source Res=256 Batch=32: 1.781 ms/img | 561.5 FPS

Resolution 384



Resolution 384 eval:  99%|█████████▉| 174/175 [01:06<00:00,  2.52it/s, ms_per_img=1.738, samples=5,597]
                                                                                                       

Acc=0.8571 AUC=0.5575 Fast=100.00% Medium=0.00% Full=0.00%


Resolution sweep: 100%|██████████| 4/4 [07:06<00:00, 106.58s/it]

Source Res=384 Single: 5.859 ms/img | 170.7 FPS
Source Res=384 Batch=32: 1.781 ms/img | 561.4 FPS

====================================MaD-CoRN EXPERIMENT RESULTS=====================================

[1] TRAINING PERIODS RESULTS
                                               Training Periods Results                                              

+-------+------------+-----------+-----------+---------+---------+-----------+--------+--------+--------------------+
| Epoch | Train Loss | Train Acc | Train AUC | Val Acc | Val AUC | Precision | Recall | F1     | Uncertain Rate (%) |
+-------+------------+-----------+-----------+---------+---------+-----------+--------+--------+--------------------+
| 1     | 0.7403     | 0.1444    | 0.4939    | 0.1429  | 0.4981  | 0.0000    | 0.0000 | 0.0000 | 100.00             |
| 2     | 0.4496     | 0.8565    | 0.5112    | 0.8571  | 0.5343  | 0.8571    | 1.0000 | 0.9231 | 0.00               |
| 3     | 0.4514     | 0.8570    | 0.5214    | 0.8571  | 0.53

In [1]:
print('test')

test


In [2]:
!pip install scikit-learn torch torchvision

In [3]:
!pip install kagglehub
import kagglehub

# Download latest version
path = kagglehub.dataset_download("pranabr0y/celebdf-v2image-dataset")

print("Path to dataset files:", path)

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /home/jovyan/.cache/kagglehub/datasets/pranabr0y/celebdf-v2image-dataset/versions/1


In [4]:
# CELL 1 — Imports

import io
import os
import csv
import copy
import json
import math
import time
import random
import warnings
import numpy as np
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Any, Optional

warnings.filterwarnings('ignore')

from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import (
    Dataset, DataLoader, WeightedRandomSampler, TensorDataset
)
import torchvision.transforms as T
from torch.cuda.amp import GradScaler, autocast

from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score,
    f1_score, accuracy_score,
)

print("✓ All imports OK")
print(f"  PyTorch  : {torch.__version__}")
print(f"  CUDA     : {torch.cuda.is_available()}")



from tqdm.auto import tqdm


✓ All imports OK
  PyTorch  : 2.11.0+cu130
  CUDA     : True


In [5]:
# CELL 2 — Configuration  ← EDIT THIS

CONFIG = dict(
    # ── Data ──────────────────────────────────────────────────────────────────
    data_dir      = "/home/jovyan/.cache/kagglehub/datasets/pranabr0y/celebdf-v2image-dataset/versions/1/Celeb_V2",   # Celeb-DF root. Prefer Celeb_V2 with Train/ Val/ Test
    csv_path      = None,                  # optional CSV manifest (path,label)
    resolutions   = [128, 224, 256, 384], # every split is expanded across all listed resolutions
    train_res     = 224,                   # kept for smoke tests / naming, not single-res training anymore
    max_per_class = None,                  # int to cap dataset size per class before splitting, None = all
    split_ratios  = (0.80, 0.10, 0.10),   # used only when Train/Val/Test folders are not present

    # ── Training ──────────────────────────────────────────────────────────────
    epochs          = 30,
    batch_size      = 32,
    lr              = 1e-3,
    weight_decay    = 1e-4,
    label_smoothing = 0.05,
    warmup_epochs   = 3,
    patience        = 8,

    # ── Model ─────────────────────────────────────────────────────────────────
    base_ch   = 32,

    # ── System ────────────────────────────────────────────────────────────────
    num_workers = 4,
    seed        = 42,
    save_dir    = "./mad_corn_results",
    device      = "auto",

    # ── Dev mode ──────────────────────────────────────────────────────────────
    smoke_test  = False,
)



In [6]:
# CELL 3 — Utility helpers

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def auto_device(requested: str) -> torch.device:
    if requested != "auto":
        return torch.device(requested)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

set_seed(CONFIG["seed"])
DEVICE = auto_device(CONFIG["device"])
os.makedirs(CONFIG["save_dir"], exist_ok=True)
print(f"✓ Device  : {DEVICE}")
print(f"  Save dir: {CONFIG['save_dir']}")




✓ Device  : cuda
  Save dir: ./mad_corn_results


In [7]:
# CELL 4 — Result dataclasses

@dataclass
class EpochResult:
    epoch:          int
    train_loss:     float
    train_acc:      float
    train_auc:      float
    val_acc:        float
    val_auc:        float
    precision:      float
    recall:         float
    f1:             float
    uncertain_rate: float
    lr:             float

@dataclass
class TestResult:
    model_name:     str
    accuracy:       float
    auc:            float
    precision:      float
    recall:         float
    f1:             float
    uncertain_rate: float

@dataclass
class ResolutionResult:
    resolution:  int
    accuracy:    float
    auc:         float
    fast_pct:    float   # % batches with <5 ms per-image latency
    medium_pct:  float   # 5–20 ms
    full_pct:    float   # >20 ms

@dataclass
class InferenceResult:
    mode:              str
    time_per_image_ms: float
    fps:               float

@dataclass
class TrainingTimeResult:
    model_type: str
    minutes:    float
    hours:      float

print("✓ Dataclasses defined")




✓ Dataclasses defined


In [8]:
# CELL 5 — Model: MaD-CoRN Architecture

# ── Utility Blocks ──────────────────────────────────────────────────────────────

class ConvBNReLU(nn.Module):
    """Standard Conv → BN → ReLU6 block."""
    def __init__(self, in_ch, out_ch, kernel=3, stride=1, padding=1, groups=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel, stride, padding, groups=groups, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU6(inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class DepthwiseSeparable(nn.Module):
    """Depthwise Separable Convolution — core efficiency component."""
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.dw = ConvBNReLU(in_ch, in_ch, kernel=3, stride=stride, padding=1, groups=in_ch)
        self.pw = ConvBNReLU(in_ch, out_ch, kernel=1, padding=0)
    def forward(self, x):
        return self.pw(self.dw(x))


# ── Convolutional Reservoir (CoRN) ──────────────────────────────────────────────

class ConvReservoir(nn.Module):
    """
    Echo State Network-inspired convolutional reservoir.
    - Fixed (frozen) random reservoir weights W_res
    - Trainable input projection + output readout
    - Reservoir update: h_res = tanh(W_res * h)
    - Combined via residual: h + h_res
    """
    def __init__(self, in_ch, reservoir_ch=64, spectral_radius=0.9):
        super().__init__()
        self.reservoir_ch = reservoir_ch

        # Trainable input projection
        self.input_proj = nn.Conv2d(in_ch, reservoir_ch, kernel_size=1, bias=False)
        self.input_bn   = nn.BatchNorm2d(reservoir_ch)

        # Fixed reservoir (registered as buffer → not a parameter → not trained)
        W = torch.randn(reservoir_ch, reservoir_ch, 3, 3)
        W = W / (W.abs().max() + 1e-8) * spectral_radius
        self.register_buffer("W_res", W)

        # Trainable readout
        self.readout = nn.Sequential(
            nn.Conv2d(reservoir_ch, reservoir_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(reservoir_ch),
            nn.ReLU6(inplace=True),
        )

    def forward(self, x):
        h     = F.relu(self.input_bn(self.input_proj(x)))
        h_res = torch.tanh(F.conv2d(h, self.W_res, padding=1))  # reservoir echo
        return self.readout(h + h_res)                            # residual combine


# ── Multi-scale Attention (MaD) ─────────────────────────────────────────────────

class ChannelAttention(nn.Module):
    """Squeeze-and-Excitation channel attention."""
    def __init__(self, channels, reduction=4):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, mid),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels),
            nn.Sigmoid(),
        )
    def forward(self, x):
        w = self.gate(x).view(x.size(0), x.size(1), 1, 1)
        return x * w


class SpatialAttention(nn.Module):
    """CBAM-style spatial attention."""
    def __init__(self):
        super().__init__()
        self.conv    = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx, _ = x.max(dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))


class MultiScaleAttentionBlock(nn.Module):
    """
    MaD block: 3 parallel branches at different depths → fuse → dual attention.
    Branch depths 1/2/3 give multi-scale receptive fields for artefact capture.
    """
    def __init__(self, in_ch, out_ch):
        super().__init__()
        b_ch = out_ch // 3

        self.branch1 = DepthwiseSeparable(in_ch, b_ch)
        self.branch2 = nn.Sequential(
            DepthwiseSeparable(in_ch, b_ch),
            DepthwiseSeparable(b_ch,  b_ch),
        )
        self.branch3 = nn.Sequential(
            DepthwiseSeparable(in_ch, b_ch),
            DepthwiseSeparable(b_ch,  b_ch),
            DepthwiseSeparable(b_ch,  b_ch),
        )
        self.fuse = ConvBNReLU(b_ch * 3, out_ch, kernel=1, padding=0)
        self.ca   = ChannelAttention(out_ch)
        self.sa   = SpatialAttention()

    def forward(self, x):
        f = self.fuse(torch.cat([self.branch1(x), self.branch2(x), self.branch3(x)], dim=1))
        return self.sa(self.ca(f))


# ── Full MaD-CoRN Model ─────────────────────────────────────────────────────────

class MaDCoRN(nn.Module):
    """
    MaD-CoRN: Multi-scale Attention Detection + Convolutional Reservoir Network.

    Stages: Stem → [MaD×2 + CoRN + Downsample] × 3 → GlobalAvgPool → Classifier
    ~3–5 M parameters (base_ch=32), resolution-agnostic (tested 128–384).
    """
    def __init__(self, in_ch=3, num_classes=2, base_ch=32):
        super().__init__()
        c = base_ch

        # Stem
        self.stem = nn.Sequential(
            ConvBNReLU(in_ch, c,     kernel=3, stride=2, padding=1),
            ConvBNReLU(c,    c * 2,  kernel=3, stride=1, padding=1),
        )
        # Stage 1
        self.mad1  = MultiScaleAttentionBlock(c * 2,  c * 4)
        self.mad2  = MultiScaleAttentionBlock(c * 4,  c * 4)
        self.res1  = ConvReservoir(c * 4, c * 4)
        self.down1 = nn.Conv2d(c * 4, c * 4, kernel_size=3, stride=2, padding=1, bias=False)
        # Stage 2
        self.mad3  = MultiScaleAttentionBlock(c * 4,  c * 8)
        self.mad4  = MultiScaleAttentionBlock(c * 8,  c * 8)
        self.res2  = ConvReservoir(c * 8, c * 8)
        self.down2 = nn.Conv2d(c * 8, c * 8, kernel_size=3, stride=2, padding=1, bias=False)
        # Stage 3
        self.mad5  = MultiScaleAttentionBlock(c * 8,  c * 16)
        self.mad6  = MultiScaleAttentionBlock(c * 16, c * 16)
        self.res3  = ConvReservoir(c * 16, c * 16)
        # Classifier
        self.pool  = nn.AdaptiveAvgPool2d(1)
        self.head  = nn.Sequential(
            nn.Flatten(),
            nn.Linear(c * 16, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x)
        x = self.down1(self.res1(self.mad2(self.mad1(x))))
        x = self.down2(self.res2(self.mad4(self.mad3(x))))
        x = self.res3(self.mad6(self.mad5(x)))
        return self.head(self.pool(x))

    def count_parameters(self):
        total  = sum(p.numel() for p in self.parameters())
        frozen = sum(p.numel() for p in self.parameters() if not p.requires_grad)
        return total, frozen


def freeze_reservoir_weights(model: MaDCoRN):
    """Buffers are already non-trainable, but also guard any named params."""
    for name, param in model.named_parameters():
        if "W_res" in name:
            param.requires_grad_(False)


# Quick sanity check
_m = MaDCoRN(base_ch=CONFIG["base_ch"])
freeze_reservoir_weights(_m)
_total, _frozen = _m.count_parameters()
print(f"✓ MaD-CoRN instantiated")
print(f"  Total params : {_total:,}")
print(f"  Frozen params: {_frozen:,}")
for _r in CONFIG["resolutions"]:
    _x = torch.randn(1, 3, _r, _r)
    _o = _m(_x)
    print(f"  {_r}×{_r} → {_o.shape}  ✓")
del _m, _x, _o




✓ MaD-CoRN instantiated
  Total params : 3,428,062
  Frozen params: 0
  128×128 → torch.Size([1, 2])  ✓
  224×224 → torch.Size([1, 2])  ✓
  256×256 → torch.Size([1, 2])  ✓
  384×384 → torch.Size([1, 2])  ✓


In [9]:
# CELL 6 — Dataset & DataLoaders

# ── Augmentation helpers ────────────────────────────────────────────────────────

class AddGaussianNoise:
    def __init__(self, std=0.01):
        self.std = std
    def __call__(self, t: torch.Tensor) -> torch.Tensor:
        return t + torch.randn_like(t) * self.std


class JPEGCompression:
    """Simulate JPEG compression artefacts (train-time aug)."""
    def __init__(self, quality_range=(60, 95)):
        self.lo, self.hi = quality_range
    def __call__(self, img: Image.Image) -> Image.Image:
        q   = random.randint(self.lo, self.hi)
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=q)
        buf.seek(0)
        return Image.open(buf).copy()


# ── Transform factory ───────────────────────────────────────────────────────────

_MEAN = [0.485, 0.456, 0.406]
_STD  = [0.229, 0.224, 0.225]

def build_transforms(resolution: int, split: str = "train") -> T.Compose:
    if split == "train":
        return T.Compose([
            T.Resize((resolution + 16, resolution + 16)),
            T.RandomCrop(resolution),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.1),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
            T.RandomGrayscale(p=0.05),
            T.RandomApply([T.GaussianBlur(kernel_size=3)], p=0.15),
            JPEGCompression(quality_range=(60, 95)),
            T.ToTensor(),
            AddGaussianNoise(std=0.01),
            T.Normalize(_MEAN, _STD),
        ])
    return T.Compose([
        T.Resize((resolution, resolution)),
        T.ToTensor(),
        T.Normalize(_MEAN, _STD),
    ])


# ── Celeb-DF path helpers ──────────────────────────────────────────────────────

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
REAL_DIRS  = {"real", "celeb-real", "youtube-real", "original"}
FAKE_DIRS  = {"fake", "celeb-synthesis", "synthesis", "manipulated"}

def _detect_label(dirname: str) -> Optional[int]:
    n = dirname.lower()
    if any(n == d or n.startswith(d) for d in REAL_DIRS):
        return 0
    if any(n == d or n.startswith(d) for d in FAKE_DIRS):
        return 1
    return None

def gather_labeled_paths(root: str) -> list[tuple[str, int]]:
    root = Path(root)
    samples = []
    for sub in root.iterdir():
        if not sub.is_dir():
            continue
        label = _detect_label(sub.name)
        if label is None:
            continue
        for p in sub.rglob("*"):
            if p.suffix.lower() in IMAGE_EXTS:
                samples.append((str(p), label))
    return sorted(samples)

def load_csv_samples(csv_path: str) -> list[tuple[str, int]]:
    samples = []
    with open(csv_path, newline="") as f:
        for row in csv.DictReader(f):
            p = row.get("path") or row.get("file") or list(row.values())[0]
            l = int(row.get("label", row.get("Label", 0)))
            if Path(p).suffix.lower() in IMAGE_EXTS and os.path.exists(p):
                samples.append((p, l))
    return samples

def cap_per_class(samples: list[tuple[str, int]], max_per_class: Optional[int], seed: int) -> list[tuple[str, int]]:
    if not max_per_class:
        return samples
    rng = random.Random(seed)
    real = [(p, l) for p, l in samples if l == 0]
    fake = [(p, l) for p, l in samples if l == 1]
    real = rng.sample(real, min(len(real), max_per_class))
    fake = rng.sample(fake, min(len(fake), max_per_class))
    out = real + fake
    rng.shuffle(out)
    return out

def split_samples(samples: list[tuple[str, int]], ratios=(0.8, 0.1, 0.1), seed: int = 42):
    tr_ratio, val_ratio, te_ratio = ratios
    if not math.isclose(tr_ratio + val_ratio + te_ratio, 1.0, rel_tol=1e-6, abs_tol=1e-6):
        raise ValueError(f"split_ratios must sum to 1.0, got {ratios}")

    rng = random.Random(seed)
    by_label = {0: [], 1: []}
    for item in samples:
        by_label[item[1]].append(item)

    split_map = {"train": [], "val": [], "test": []}
    for label, items in by_label.items():
        rng.shuffle(items)
        n = len(items)
        n_tr = int(n * tr_ratio)
        n_val = int(n * val_ratio)
        n_te = n - n_tr - n_val
        split_map["train"].extend(items[:n_tr])
        split_map["val"].extend(items[n_tr:n_tr+n_val])
        split_map["test"].extend(items[n_tr+n_val:])

    for k in split_map:
        rng.shuffle(split_map[k])
    return split_map

def load_celebdf_splits(cfg: dict) -> dict[str, list[tuple[str, int]]]:
    data_dir = Path(cfg["data_dir"])
    seed = cfg["seed"]
    max_per_class = cfg["max_per_class"]

    if cfg.get("csv_path"):
        base_samples = cap_per_class(load_csv_samples(cfg["csv_path"]), max_per_class, seed)
        return split_samples(base_samples, ratios=cfg["split_ratios"], seed=seed)

    split_dirs = {name: data_dir / name for name in ["Train", "Val", "Test"]}
    has_explicit_splits = all(p.exists() and p.is_dir() for p in split_dirs.values())

    if has_explicit_splits:
        split_map = {
            "train": cap_per_class(gather_labeled_paths(split_dirs["Train"]), max_per_class, seed),
            "val":   cap_per_class(gather_labeled_paths(split_dirs["Val"]),   max_per_class, seed),
            "test":  cap_per_class(gather_labeled_paths(split_dirs["Test"]),  max_per_class, seed),
        }
        total = sum(len(v) for v in split_map.values())
        print("Detected explicit Celeb-DF split folders: Train / Val / Test")
        print(f"Requested split_ratios={cfg['split_ratios']} ignored because folder splits already exist.")
        print(f"Using folder totals: train={len(split_map['train']):,}  val={len(split_map['val']):,}  test={len(split_map['test']):,}  total={total:,}")
        return split_map

    base_samples = cap_per_class(gather_labeled_paths(data_dir), max_per_class, seed)
    split_map = split_samples(base_samples, ratios=cfg["split_ratios"], seed=seed)
    print(f"No Train/Val/Test folders found. Built random split with ratios={cfg['split_ratios']}.")
    return split_map


# ── Datasets ───────────────────────────────────────────────────────────────────

class MultiResolutionCelebDFDataset(Dataset):
    """
    Expands every image across all requested resolutions.

    Each item is:
      (image_tensor_at_requested_resolution, label)

    Metadata for each expanded sample is kept in self.expanded_samples as:
      (path, label, resolution)
    """
    def __init__(
        self,
        base_samples: list[tuple[str, int]],
        resolutions: list[int],
        split: str = "train",
    ):
        self.base_samples = list(base_samples)
        self.resolutions = list(resolutions)
        self.split = split
        self.transforms = {res: build_transforms(res, split) for res in self.resolutions}
        self.expanded_samples = [
            (path, label, res)
            for path, label in self.base_samples
            for res in self.resolutions
        ]

        n_r = sum(1 for _, l in self.base_samples if l == 0)
        n_f = len(self.base_samples) - n_r
        print(
            f"  [{split:>5}] base: real={n_r:,}  fake={n_f:,}  total={len(self.base_samples):,} | "
            f"expanded across {len(self.resolutions)} resolutions → {len(self.expanded_samples):,}"
        )

    def __len__(self):
        return len(self.expanded_samples)

    def __getitem__(self, idx):
        path, label, resolution = self.expanded_samples[idx]
        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            img = Image.fromarray(np.zeros((resolution, resolution, 3), dtype=np.uint8))
        return self.transforms[resolution](img), label

    def subset_for_resolution(self, resolution: int):
        return SingleResolutionView(self, resolution)

    @property
    def base_count(self) -> int:
        return len(self.base_samples)

    @property
    def expanded_count(self) -> int:
        return len(self.expanded_samples)


class SingleResolutionView(Dataset):
    def __init__(self, parent: MultiResolutionCelebDFDataset, resolution: int):
        self.parent = parent
        self.resolution = resolution
        self.indices = [i for i, (_, _, r) in enumerate(parent.expanded_samples) if r == resolution]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        return self.parent[self.indices[idx]]


class ResolutionAwareBatchSampler(torch.utils.data.Sampler[list[int]]):
    """
    Keeps batches homogeneous in spatial size so tensors can stack cleanly.
    """
    def __init__(self, dataset: MultiResolutionCelebDFDataset, batch_size: int, shuffle: bool, drop_last: bool, seed: int = 42):
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.drop_last = drop_last
        self.seed = seed
        self.epoch = 0

        self.indices_by_res = {}
        for idx, (_, _, res) in enumerate(dataset.expanded_samples):
            self.indices_by_res.setdefault(res, []).append(idx)

    def __iter__(self):
        rng = random.Random(self.seed + self.epoch)
        batches = []

        for res, indices in self.indices_by_res.items():
            idxs = list(indices)
            if self.shuffle:
                rng.shuffle(idxs)

            limit = len(idxs) if not self.drop_last else (len(idxs) // self.batch_size) * self.batch_size
            idxs = idxs[:limit]

            for start in range(0, len(idxs), self.batch_size):
                batch = idxs[start:start+self.batch_size]
                if len(batch) == self.batch_size or (not self.drop_last and len(batch) > 0):
                    batches.append(batch)

        if self.shuffle:
            rng.shuffle(batches)
        return iter(batches)

    def __len__(self):
        total = 0
        for indices in self.indices_by_res.values():
            if self.drop_last:
                total += len(indices) // self.batch_size
            else:
                total += math.ceil(len(indices) / self.batch_size)
        return total

    def set_epoch(self, epoch: int):
        self.epoch = epoch


# ── DataLoader factory ──────────────────────────────────────────────────────────

def _make_loader(dataset, batch_size: int, num_workers: int, shuffle: bool, drop_last: bool, seed: int):
    batch_sampler = ResolutionAwareBatchSampler(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        seed=seed,
    )
    loader = DataLoader(
        dataset,
        batch_sampler=batch_sampler,
        num_workers=num_workers,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=(num_workers > 0),
    )
    loader.resolution_batch_sampler = batch_sampler
    return loader

def print_dataset_summary(split_datasets: dict[str, MultiResolutionCelebDFDataset], resolutions: list[int]):
    print("\nDataset size summary")
    for name in ["train", "val", "test"]:
        ds = split_datasets[name]
        print(
            f"  {name.title():<5} | base={ds.base_count:,}  "
            f"expanded={ds.expanded_count:,}  "
            f"resolutions={resolutions}"
        )

def make_dataloaders(cfg: dict) -> dict[str, Any]:
    split_map = load_celebdf_splits(cfg)
    resolutions = list(cfg["resolutions"])
    batch_size = cfg["batch_size"]
    num_workers = cfg["num_workers"]
    seed = cfg["seed"]

    split_datasets = {
        name: MultiResolutionCelebDFDataset(samples, resolutions, split=name)
        for name, samples in split_map.items()
    }
    print_dataset_summary(split_datasets, resolutions)

    loaders = {
        "train": _make_loader(split_datasets["train"], batch_size, num_workers, shuffle=True,  drop_last=True,  seed=seed),
        "val":   _make_loader(split_datasets["val"],   batch_size, num_workers, shuffle=False, drop_last=False, seed=seed),
        "test":  _make_loader(split_datasets["test"],  batch_size, num_workers, shuffle=False, drop_last=False, seed=seed),
    }
    return {
        "loaders": loaders,
        "datasets": split_datasets,
        "base_splits": split_map,
    }

def make_smoke_loader(resolution: int, n: int = 200, bs: int = 8) -> DataLoader:
    imgs   = torch.randn(n, 3, resolution, resolution)
    labels = torch.randint(0, 2, (n,))
    return DataLoader(TensorDataset(imgs, labels), batch_size=bs, shuffle=True)


print("✓ Dataset classes defined")



✓ Dataset classes defined


In [ ]:
# CELL 7 — Training engine

def compute_uncertain_rate(probs: np.ndarray, threshold: float = 0.15) -> float:
    """% of samples where |P(fake) - 0.5| < threshold → 'uncertain'."""
    fake_p = probs[:, 1] if probs.ndim == 2 else probs
    return float(np.mean(np.abs(fake_p - 0.5) < threshold) * 100)


def run_epoch(
    model, loader, criterion, optimizer, device,
    scaler=None, is_train: bool = True, desc: Optional[str] = None,
) -> dict[str, Any]:
    model.train(is_train)
    total_loss = 0.0
    seen = 0
    all_probs, all_preds, all_labels = [], [], []

    pbar = tqdm(
        loader,
        total=len(loader) if hasattr(loader, "__len__") else None,
        desc=desc or ("Train" if is_train else "Eval"),
        leave=False,
        dynamic_ncols=True,
        mininterval=0.5,
    )

    with torch.set_grad_enabled(is_train):
        for imgs, labels in pbar:
            imgs   = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            if is_train and scaler:
                with autocast():
                    logits = model(imgs)
                    loss   = criterion(logits, labels)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            else:
                logits = model(imgs)
                loss   = criterion(logits, labels)
                if is_train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)

            probs = torch.softmax(logits.float(), dim=1).detach().cpu().numpy()
            all_probs.append(probs)
            all_preds.append(probs.argmax(axis=1))
            all_labels.append(labels.cpu().numpy())
            batch_n = len(labels)
            total_loss += loss.item() * batch_n
            seen += batch_n
            pbar.set_postfix(loss=f"{loss.item():.4f}", samples=f"{seen:,}")

    all_probs  = np.vstack(all_probs)
    all_preds  = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    n          = len(all_labels)

    acc  = accuracy_score(all_labels, all_preds)
    try:
        auc = roc_auc_score(all_labels, all_probs[:, 1])
    except ValueError:
        auc = 0.5
    prec = precision_score(all_labels, all_preds, zero_division=0)
    rec  = recall_score(all_labels, all_preds, zero_division=0)
    f1   = f1_score(all_labels, all_preds, zero_division=0)
    unc  = compute_uncertain_rate(all_probs)

    return dict(loss=total_loss/n, acc=acc, auc=auc,
                precision=prec, recall=rec, f1=f1,
                uncertain_rate=unc, probs=all_probs,
                preds=all_preds, labels=all_labels)


class Trainer:
    """
    Full training loop:
      - AdamW optimiser
      - Cosine LR schedule with linear warmup
      - Mixed-precision AMP (CUDA only)
      - Label smoothing
      - Best-model checkpointing (by Val AUC)
      - Early stopping
    """
    def __init__(self, model, train_loader, val_loader, test_loader,
                 device, cfg: dict, ckpt_path: str):
        self.model        = model.to(device)
        self.train_loader = train_loader
        self.val_loader   = val_loader
        self.test_loader  = test_loader
        self.device       = device
        self.epochs       = cfg["epochs"]
        self.patience     = cfg["patience"]
        self.ckpt_path    = ckpt_path
        self.model_name   = f"MaD-CoRN-{cfg['train_res']}"

        self.criterion = nn.CrossEntropyLoss(label_smoothing=cfg["label_smoothing"])
        self.optimizer = optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=cfg["lr"], weight_decay=cfg["weight_decay"],
        )
        total_steps  = self.epochs * len(train_loader)
        warmup_steps = cfg["warmup_epochs"] * len(train_loader)

        def lr_lambda(step):
            if step < warmup_steps:
                return step / max(1, warmup_steps)
            prog = (step - warmup_steps) / max(1, total_steps - warmup_steps)
            return 0.5 * (1.0 + math.cos(math.pi * prog))

        self.scheduler = optim.lr_scheduler.LambdaLR(self.optimizer, lr_lambda)
        self.scaler    = GradScaler() if device.type == "cuda" else None
        self.epoch_results: list[EpochResult] = []

    def train(self) -> tuple[list[EpochResult], float]:
        best_auc   = 0.0
        no_improve = 0
        t0         = time.time()

        for epoch in range(1, self.epochs + 1):
            # if hasattr(self.train_loader, "resolution_batch_sampler"):
            #     self.train_loader.resolution_batch_sampler.set_epoch(epoch)
            tr = run_epoch(self.model, self.train_loader, self.criterion,
                           self.optimizer, self.device, self.scaler,
                           is_train=True, desc=f"Epoch {epoch}/{self.epochs} [train]")
            self.scheduler.step()

            with torch.no_grad():
                vl = run_epoch(self.model, self.val_loader, self.criterion,
                               None, self.device, None,
                               is_train=False, desc=f"Epoch {epoch}/{self.epochs} [val]")

            lr_now = self.optimizer.param_groups[0]["lr"]
            er = EpochResult(
                epoch=epoch,
                train_loss=tr["loss"],       train_acc=tr["acc"],  train_auc=tr["auc"],
                val_acc=vl["acc"],           val_auc=vl["auc"],
                precision=vl["precision"],   recall=vl["recall"],  f1=vl["f1"],
                uncertain_rate=vl["uncertain_rate"], lr=lr_now,
            )
            self.epoch_results.append(er)

            print(
                f"Epoch {epoch:3d}/{self.epochs} | "
                f"Loss {er.train_loss:.4f} | "
                f"Tr Acc {er.train_acc:.4f} AUC {er.train_auc:.4f} | "
                f"Val Acc {er.val_acc:.4f} AUC {er.val_auc:.4f} | "
                f"P {er.precision:.4f} R {er.recall:.4f} F1 {er.f1:.4f} | "
                f"Unc {er.uncertain_rate:.1f}% | LR {lr_now:.2e}"
            )

            if vl["auc"] > best_auc:
                best_auc = vl["auc"]
                no_improve = 0
                torch.save(self.model.state_dict(), self.ckpt_path)
                print(f"  ✓ Checkpoint saved (Val AUC = {best_auc:.4f})")
            else:
                no_improve += 1
                if no_improve >= self.patience:
                    print(f"  ⏹ Early stopping at epoch {epoch}")
                    break

        return self.epoch_results, (time.time() - t0) / 60

    def evaluate(self, loader=None, load_best: bool = True) -> TestResult:
        if load_best and os.path.exists(self.ckpt_path):
            self.model.load_state_dict(torch.load(self.ckpt_path, map_location=self.device))
        loader = loader or self.test_loader
        with torch.no_grad():
            r = run_epoch(self.model, loader, self.criterion,
                          None, self.device, None,
                          is_train=False, desc="Test evaluation")
        return TestResult(
            model_name=self.model_name,
            accuracy=r["acc"], auc=r["auc"],
            precision=r["precision"], recall=r["recall"],
            f1=r["f1"], uncertain_rate=r["uncertain_rate"],
        )


print("✓ Trainer defined")





In [11]:
# CELL 8 — Inference timing & resolution evaluation

@torch.no_grad()
def measure_inference(
    model, resolution: int, device,
    batch_size: int = 32, n_warmup: int = 20, n_runs: int = 200,
) -> list[InferenceResult]:
    """Benchmark single-image and batch inference speed."""
    model.eval()
    results = []
    for bs, label in [(1, "Single"), (batch_size, f"Batch={batch_size}")]:
        dummy = torch.randn(bs, 3, resolution, resolution, device=device)
        for _ in range(n_warmup):
            model(dummy)
        if device.type == "cuda": torch.cuda.synchronize()

        t0 = time.perf_counter()
        for _ in range(n_runs):
            model(dummy)
        if device.type == "cuda": torch.cuda.synchronize()
        elapsed = time.perf_counter() - t0

        total = bs * n_runs
        results.append(InferenceResult(
            mode=f"Res={resolution} {label}",
            time_per_image_ms=elapsed / total * 1000,
            fps=total / elapsed,
        ))
    return results


@torch.no_grad()
def evaluate_by_resolution(
    model, loader, device, resolution: int,
) -> ResolutionResult:
    """Evaluate accuracy + latency buckets (Fast <5ms, Medium 5–20ms, Full >20ms)."""
    model.eval()
    all_probs, all_labels = [], []
    buckets = {"fast": 0, "medium": 0, "full": 0}
    total   = 0

    pbar = tqdm(
        loader,
        total=len(loader) if hasattr(loader, "__len__") else None,
        desc=f"Resolution {resolution} eval",
        leave=False,
        dynamic_ncols=True,
        mininterval=0.5,
    )

    for imgs, labels in pbar:
        imgs = imgs.to(device, non_blocking=True)
        bs   = imgs.size(0)

        t0     = time.perf_counter()
        logits = model(imgs)
        if device.type == "cuda": torch.cuda.synchronize()
        ms = (time.perf_counter() - t0) / bs * 1000

        probs = torch.softmax(logits.float(), dim=1).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.numpy())

        if ms < 5:    buckets["fast"]   += bs
        elif ms < 20: buckets["medium"] += bs
        else:         buckets["full"]   += bs
        total += bs
        pbar.set_postfix(samples=f"{total:,}", ms_per_img=f"{ms:.3f}")

    all_probs  = np.vstack(all_probs)
    all_labels = np.concatenate(all_labels)
    preds      = all_probs.argmax(axis=1)

    acc = accuracy_score(all_labels, preds)
    try: auc = roc_auc_score(all_labels, all_probs[:, 1])
    except ValueError: auc = 0.5

    return ResolutionResult(
        resolution=resolution, accuracy=acc, auc=auc,
        fast_pct  =buckets["fast"]   / max(total, 1) * 100,
        medium_pct=buckets["medium"] / max(total, 1) * 100,
        full_pct  =buckets["full"]   / max(total, 1) * 100,
    )


print("✓ Inference / resolution evaluation functions defined")





✓ Inference / resolution evaluation functions defined


In [12]:
# CELL 9 — Results reporting (tables + file saving)

def _table(headers: list, rows: list, title: str = "") -> str:
    col_w = [max(len(str(h)), max((len(str(r[i])) for r in rows), default=0))
             for i, h in enumerate(headers)]
    sep  = "+" + "+".join("-" * (w + 2) for w in col_w) + "+"
    hdr  = "|" + "|".join(f" {h:<{w}} " for h, w in zip(headers, col_w)) + "|"
    body = ["|" + "|".join(f" {str(v):<{w}} " for v, w in zip(r, col_w)) + "|"
            for r in rows]
    out  = [sep, hdr, sep] + body + [sep]
    if title:
        out.insert(0, title.center(len(sep)))
        out.insert(1, "")
    return "\n".join(out)


def print_training_table(ers: list[EpochResult]) -> str:
    h = ["Epoch","Train Loss","Train Acc","Train AUC",
         "Val Acc","Val AUC","Precision","Recall","F1","Uncertain Rate (%)"]
    rows = [[e.epoch, f"{e.train_loss:.4f}", f"{e.train_acc*100:.2f}%",
             f"{e.train_auc:.4f}", f"{e.val_acc*100:.2f}%", f"{e.val_auc:.4f}",
             f"{e.precision:.4f}", f"{e.recall:.4f}", f"{e.f1:.4f}",
             f"{e.uncertain_rate:.2f}"] for e in ers]
    t = _table(h, rows, "Training Periods Results")
    print(t); return t


def print_test_table(trs: list[TestResult]) -> str:
    h = ["Model","Accuracy","AUC","Precision","Recall","F1","Uncertain Rate (%)"]
    rows = [[r.model_name, f"{r.accuracy*100:.2f}%", f"{r.auc:.4f}",
             f"{r.precision:.4f}", f"{r.recall:.4f}", f"{r.f1:.4f}",
             f"{r.uncertain_rate:.2f}"] for r in trs]
    t = _table(h, rows, "Test Results")
    print(t); return t


def print_resolution_table(rrs: list[ResolutionResult]) -> str:
    h = ["Resolution","Accuracy","AUC","Fast (%)","Medium (%)","Full (%)"]
    rows = [[f"{r.resolution}×{r.resolution}", f"{r.accuracy*100:.2f}%",
             f"{r.auc:.4f}", f"{r.fast_pct:.1f}", f"{r.medium_pct:.1f}",
             f"{r.full_pct:.1f}"] for r in rrs]
    t = _table(h, rows, "Accuracy Based on Resolution")
    print(t); return t


def print_inference_table(irs: list[InferenceResult]) -> str:
    h = ["Mode","Time per Image (ms)","FPS"]
    rows = [[r.mode, f"{r.time_per_image_ms:.3f}", f"{r.fps:.1f}"] for r in irs]
    t = _table(h, rows, "Inference Time")
    print(t); return t


def print_training_time_table(tts: list[TrainingTimeResult]) -> str:
    h = ["Model Type","min","hrs"]
    rows = [[r.model_type, f"{r.minutes:.1f}", f"{r.hours:.2f}"] for r in tts]
    t = _table(h, rows, "Training Time")
    print(t); return t


def save_all_results(epoch_results, test_results, res_results, inf_results,
                     tt_results, save_dir: str):
    os.makedirs(save_dir, exist_ok=True)
    data = dict(
        epoch_results=[asdict(e) for e in epoch_results],
        test_results =[asdict(t) for t in test_results],
        res_results  =[asdict(r) for r in res_results],
        inference    =[asdict(i) for i in inf_results],
        training_time=[asdict(t) for t in tt_results],
    )
    # JSON
    with open(f"{save_dir}/mad_corn_results.json", "w") as f:
        json.dump(data, f, indent=2)
    # CSVs
    for name, lst in data.items():
        if not lst: continue
        with open(f"{save_dir}/mad_corn_{name}.csv", "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=lst[0].keys())
            w.writeheader(); w.writerows(lst)
    print(f"\n✓ Results saved to: {save_dir}/")


def print_full_report(epoch_results, test_results, res_results,
                      inf_results, tt_results, save_dir):
    div = "=" * 100
    print(f"\n{div}")
    print(" MaD-CoRN EXPERIMENT RESULTS ".center(100, "="))
    print(f"{div}\n")
    print("\n[1] TRAINING PERIODS RESULTS")
    print_training_table(epoch_results)
    print("\n[2] TEST RESULTS")
    print_test_table(test_results)
    print("\n[3] ACCURACY BASED ON RESOLUTION")
    print_resolution_table(res_results)
    print("\n[4] INFERENCE TIME")
    print_inference_table(inf_results)
    print("\n[5] TRAINING TIME")
    print_training_time_table(tt_results)
    print(f"\n{div}\n")
    save_all_results(epoch_results, test_results, res_results,
                     inf_results, tt_results, save_dir)


print("✓ Reporting functions defined")




✓ Reporting functions defined


In [13]:
# CELL 10 — RUN THE FULL PIPELINE

def run_pipeline(cfg: dict):
    set_seed(cfg["seed"])
    device = auto_device(cfg["device"])
    os.makedirs(cfg["save_dir"], exist_ok=True)

    print(f'\n{"="*70}')
    print(f'  MaD-CoRN Pipeline')
    print(f'  Device      : {device}')
    print(f'  Resolutions : {cfg["resolutions"]}')
    print(f'  Split ratios: {cfg["split_ratios"]}')
    print(f'  Epochs      : {cfg["epochs"]}')
    print(f'  Smoke test  : {cfg["smoke_test"]}')
    print(f'{"="*70}\n')

    # ── Model ─────────────────────────────────────────────────────────────────
    model = MaDCoRN(in_ch=3, num_classes=2, base_ch=cfg["base_ch"])
    freeze_reservoir_weights(model)
    total, frozen = model.count_parameters()
    print(f"Model params: {total:,} total | {frozen:,} frozen\n")

    # ── Data ──────────────────────────────────────────────────────────────────
    split_datasets = None
    if cfg["smoke_test"]:
        print(">>> SMOKE TEST — synthetic data <<<\n")
        bs = cfg["batch_size"]
        train_loader = make_smoke_loader(cfg["train_res"], n=320, bs=bs)
        val_loader   = make_smoke_loader(cfg["train_res"], n=80,  bs=bs)
        test_loader  = make_smoke_loader(cfg["train_res"], n=80,  bs=bs)
        cfg = {**cfg, "epochs": min(cfg["epochs"], 3)}
    else:
        print("Loading Celeb-DF dataset...")
        data_bundle   = make_dataloaders(cfg)
        split_datasets = data_bundle["datasets"]
        train_loader  = data_bundle["loaders"]["train"]
        val_loader    = data_bundle["loaders"]["val"]
        test_loader   = data_bundle["loaders"]["test"]

    ckpt_path = os.path.join(cfg["save_dir"], f'mad_corn_multires.pt')

    # ── Training ──────────────────────────────────────────────────────────────
    print("\n" + "─"*60)
    print("TRAINING")
    print("─"*60)
    trainer = Trainer(model, train_loader, val_loader, test_loader,
                      device, cfg, ckpt_path)
    epoch_results, training_min = trainer.train()
    print(f"\nTraining done: {training_min:.1f} min  ({training_min/60:.2f} hrs)")

    # ── Primary test ──────────────────────────────────────────────────────────
    print("\n" + "─"*60)
    print("TEST EVALUATION ON COMBINED MULTI-RES TEST SET")
    print("─"*60)
    test_result  = trainer.evaluate(test_loader, load_best=True)
    test_results = [test_result]

    # ── Per-resolution evaluation ─────────────────────────────────────────────
    print("\n" + "─"*60)
    print("PER-RESOLUTION BREAKDOWN")
    print("─"*60)
    res_results = []
    inf_results = []

    for res in tqdm(cfg["resolutions"], desc="Resolution sweep", dynamic_ncols=True):
        print(f"\n  → Resolution {res}×{res}")
        if cfg["smoke_test"]:
            res_loader = make_smoke_loader(res, n=80, bs=cfg["batch_size"])
        else:
            res_ds = split_datasets["test"].subset_for_resolution(res)
            res_loader = DataLoader(
                res_ds,
                batch_size=cfg["batch_size"],
                shuffle=False,
                num_workers=cfg["num_workers"],
                pin_memory=(device.type == "cuda"),
                persistent_workers=(cfg["num_workers"] > 0),
            )

        rr = evaluate_by_resolution(model, res_loader, device, res)
        res_results.append(rr)
        print(f"    Acc={rr.accuracy*100:.2f}%  AUC={rr.auc:.4f}  "
              f"Fast={rr.fast_pct:.1f}%  Med={rr.medium_pct:.1f}%  Full={rr.full_pct:.1f}%")

        infs = measure_inference(model, res, device, batch_size=cfg["batch_size"])
        inf_results.extend(infs)
        for ir in infs:
            print(f"    [{ir.mode}]  {ir.time_per_image_ms:.3f} ms/img  |  {ir.fps:.1f} FPS")

    # ── Training time ─────────────────────────────────────────────────────────
    tt_results = [TrainingTimeResult(
        model_type='MaD-CoRN (multi-resolution training)',
        minutes=training_min,
        hours=training_min / 60,
    )]

    # ── Final report ──────────────────────────────────────────────────────────
    print_full_report(
        epoch_results, test_results, res_results,
        inf_results, tt_results, cfg["save_dir"],
    )

    return dict(
        epoch_results=epoch_results,
        test_results=test_results,
        res_results=res_results,
        inf_results=inf_results,
        tt_results=tt_results,
        model=model,
        trainer=trainer,
        split_datasets=split_datasets,
    )



In [ ]:
# CELL 11 — EXECUTE

# Edit CONFIG at the top of this file, then run this cell.
#
# Quick smoke test (no real data):
#   CONFIG["smoke_test"] = True
#
# Full run:
#   CONFIG["smoke_test"] = False
#   CONFIG["data_dir"]   = "/your/celeb_df/path"

results = run_pipeline(CONFIG)


  MaD-CoRN Pipeline
  Device      : cuda
  Resolutions : [128, 224, 256, 384]
  Split ratios: (0.8, 0.1, 0.1)
  Epochs      : 30
  Smoke test  : False

Model params: 3,428,062 total | 0 frozen

Loading Celeb-DF dataset...
Detected explicit Celeb-DF split folders: Train / Val / Test
Requested split_ratios=(0.8, 0.1, 0.1) ignored because folder splits already exist.
Using folder totals: train=80,824  val=10,104  test=10,103  total=101,031
  [train] base: real=40,288  fake=40,536  total=80,824 | expanded across 4 resolutions → 323,296
  [  val] base: real=5,036  fake=5,068  total=10,104 | expanded across 4 resolutions → 40,416
  [ test] base: real=5,036  fake=5,067  total=10,103 | expanded across 4 resolutions → 40,412

Dataset size summary
  Train | base=80,824  expanded=323,296  resolutions=[128, 224, 256, 384]
  Val   | base=10,104  expanded=40,416  resolutions=[128, 224, 256, 384]
  Test  | base=10,103  expanded=40,412  resolutions=[128, 224, 256, 384]

──────────────────────────────

Epoch   1/30 | Loss 0.6957 | Tr Acc 0.4986 AUC 0.5009 | Val Acc 0.4984 AUC 0.5027 | P 0.0000 R 0.0000 F1 0.0000 | Unc 100.0% | LR 3.30e-08
  ✓ Checkpoint saved (Val AUC = 0.5027)


Epoch   2/30 | Loss 0.6949 | Tr Acc 0.4983 AUC 0.5016 | Val Acc 0.4984 AUC 0.5065 | P 0.0000 R 0.0000 F1 0.0000 | Unc 100.0% | LR 6.60e-08
  ✓ Checkpoint saved (Val AUC = 0.5065)


Epoch   3/30 | Loss 0.6938 | Tr Acc 0.4997 AUC 0.5014 | Val Acc 0.4984 AUC 0.5112 | P 0.0000 R 0.0000 F1 0.0000 | Unc 100.0% | LR 9.90e-08
  ✓ Checkpoint saved (Val AUC = 0.5112)


Epoch   4/30 | Loss 0.6934 | Tr Acc 0.5010 AUC 0.5023 | Val Acc 0.5071 AUC 0.5144 | P 0.5191 R 0.2351 F1 0.3236 | Unc 100.0% | LR 1.32e-07
  ✓ Checkpoint saved (Val AUC = 0.5144)


...

Epoch  18/30 | Loss 0.6848 | Tr Acc 0.5556 AUC 0.5731 | Val Acc 0.5264 AUC 0.5689 | P 0.5605 R 0.2581 F1 0.3535 | Unc 90.3% | LR 5.94e-07


Epoch  19/30 | Loss 0.6844 | Tr Acc 0.5568 AUC 0.5752 | Val Acc 0.5314 AUC 0.5797 | P 0.5779 R 0.2438 F1 0.3429 | Unc 88.1% | LR 6.27e-07
  ✓ Checkpoint saved (Val AUC = 0.5797)


Epoch 21/30 [train]:  83%|████████▎ | 8362/10100 [30:34<07:03,  4.11it/s, loss=0.7097, samples=267,648]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

Epoch 24/30 [train]:  23%|██▎       | 2331/10100 [08:27<21:33,  6.01it/s, loss=0.6907, samples=74,656]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

Epoch 24/30 [train]:  49%|████▊     | 4923/10100 [17:54<18:23,  4.69it/s, loss=0.7226, samples=157,536]IOPub message rate exceeded.
The Jupyter server will tem

Epoch  29/30 | Loss 0.6795 | Tr Acc 0.5703 AUC 0.5950 | Val Acc 0.5651 AUC 0.5905 | P 0.5548 R 0.6731 F1 0.6082 | Unc 88.2% | LR 9.57e-07


Epoch 30/30 [train]:  54%|█████▍    | 5479/10100 [20:07<19:31,  3.94it/s, loss=0.7363, samples=175,392]